# Hybrid Movie Recommender

Refactored experiment notebook for rating prediction and top-k recommendation. Reusable code lives under `src/hybrid_movie_recommender/`.


In this project, you will work to build different recommendation models and evaluate the effectiveness of these models through offline experiments. The dataset used for the experiments is **MovieLens100K**, a movie recommendation dataset collected by GroupLens: https://grouplens.org/datasets/movielens/100k/. For more details, check the project description on Brightspace.

# Instruction

**Submission:** Answer all the questions in this jupyter-notebook file. Submit this jupyter-notebook file (your answers included) to Brightspace. Change the name of this jupyter-notebook file to your group number: example, group10 -> 10.ipynb.

# Setup

In [ ]:
# Optional in notebooks: install dependencies from the project root.
# %pip install -r ../requirements.txt


In [ ]:
# For BERT embeddings (install: pip install transformers torch)
print("Check the status of BERT installation:")

try:
    from transformers import AutoTokenizer, AutoModel
    import torch
    BERT_AVAILABLE = True
    print("BERT libraries loaded successfully!")
    device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
except ImportError:
    BERT_AVAILABLE = False
    print("BERT libraries not available. Install with: pip install transformers torch")

In [ ]:
import os
import re
import math
import time
import random
import warnings
from itertools import product
from typing import List, Tuple, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from scipy.sparse import csr_matrix
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.model_selection import train_test_split
from typing import Dict, Tuple, List, Optional
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity


from IPython.display import display
import inspect

warnings.filterwarnings('ignore')
os.makedirs('../outputs/preds', exist_ok=True)
np.random.seed(10)

print("Libraries imported successfully!")


# Load dataset

In [ ]:
from pathlib import Path

DATA_DIR = Path('../data')
columns_name = ['user_id', 'item_id', 'rating', 'timestamp']
train_data = pd.read_csv(DATA_DIR / 'training.txt', sep='\t', names=columns_name)
test_data = pd.read_csv(DATA_DIR / 'test.txt', sep='\t', names=columns_name)

# Use a held-out validation split from the original training data for sweeps.
train_data, val_data = train_test_split(
    train_data, test_size=0.2, random_state=10, stratify=train_data['user_id']
)
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

print(f'Train: {len(train_data):,} | Validation: {len(val_data):,} | Test: {len(test_data):,}')


In [ ]:
movies = pd.read_csv(DATA_DIR / 'movies.txt', names=['item_id', 'title', 'genres', 'description'], sep='\t')
movies.head()


## Task 0) Utilities

### Save Predictions

In [ ]:
def save_predictions_to_csv(
    model,
    test_data: pd.DataFrame,
    output_filepath: str,
    model_name: str = "Model") -> pd.DataFrame:
    """
    Generate predictions for test data and save to CSV.
    """
    print(f"Generating predictions using {model_name}...")
    
    predictions_list = []
    
    for _, row in tqdm(test_data[['user_id', 'item_id', 'rating']].iterrows(), 
                       total=len(test_data), 
                       desc=f"{model_name} Predictions"):
        user_id = row['user_id']
        item_id = row['item_id']
        actual_rating = row['rating']
        
        predicted_rating = model.predict_rating(user_id, item_id)
        predicted_rating = float(np.clip(predicted_rating, 1.0, 5.0))
        
        predictions_list.append({
            'user_id': user_id,
            'item_id': item_id,
            'actual_rating': float(actual_rating),
            'predicted_rating': predicted_rating
        })
    
    predictions_df = pd.DataFrame(predictions_list)
    
    rmse = np.sqrt(mean_squared_error(predictions_df['actual_rating'], 
                                       predictions_df['predicted_rating']))
    mae = mean_absolute_error(predictions_df['actual_rating'], 
                               predictions_df['predicted_rating'])
    
    predictions_df.to_csv(output_filepath, index=False)
    
    print(f"\nPredictions saved to: {output_filepath}")
    print(f"Results for {model_name}:")
    print(f"   - Total predictions: {len(predictions_df)}")
    print(f"   - RMSE: {rmse:.4f}")
    print(f"   - MAE: {mae:.4f}")

    return predictions_df

### Calculate Metrics

In [ ]:
def evaluate_rmse(model, test_data: pd.DataFrame) -> float:
    preds = []
    trues = []
    
    for _, row in tqdm(test_data[['user_id', 'item_id', 'rating']].iterrows(), total=len(test_data)):
        u = row['user_id']
        i = row['item_id']
        true_r = row['rating']
        
        pred = model.predict_rating(u, i)
        
        preds.append(float(pred))
        trues.append(float(true_r))
    
    return float(math.sqrt(mean_squared_error(trues, preds)))


def evaluate_ranking_metrics(
    model,
    test_data: pd.DataFrame,
    train_data: pd.DataFrame,
    n: int = 10,
    relevance_threshold: float = 3.0
) -> Dict[str, float]:
    """
    Fast version of evaluate_ranking_metrics:
      - Vectorized DCG/NDCG
      - Precomputes denominators
      - Minimizes Python loops
    """
    test_users = test_data['user_id'].unique()

    # Build ground-truth per user (item -> rating)
    test_group = {
        uid: grp.set_index('item_id')['rating'].to_dict()
        for uid, grp in test_data.groupby('user_id')
    }

    # Precompute discount factors for DCG@k
    discounts = 1.0 / np.log2(np.arange(2, n + 2))

    precision_vals, recall_vals, ndcg_vals = [], [], []

    for user_id in tqdm(test_users, desc=f"Fast eval@{n}"):
        gt_dict = test_group.get(user_id)
        if not gt_dict:
            continue

        relevant_items = {i for i, r in gt_dict.items() if r >= relevance_threshold}
        if not relevant_items:
            continue

        try:
            recs = model.recommend_topk(user_id, n=n)
        except Exception:
            continue

        if not recs:
            continue

        rec_items = [i for i, _ in recs[:n]]
        rels = np.array([gt_dict.get(i, 0.0) for i in rec_items], dtype=float)

        rel_hits = np.sum(np.array([i in relevant_items for i in rec_items], dtype=bool))
        precision = rel_hits / n
        recall = rel_hits / len(relevant_items)
        precision_vals.append(precision)
        recall_vals.append(recall)

        if np.any(rels):
            dcg = np.sum((2.0**rels - 1.0) * discounts)
            ideal_rels = np.sort(list(gt_dict.values()))[::-1][:n]
            idcg = np.sum((2.0**ideal_rels - 1.0) * discounts[: len(ideal_rels)])
            if idcg > 0:
                ndcg_vals.append(dcg / idcg)

    results = {
        f'precision@{n}': float(np.mean(precision_vals)) if precision_vals else 0.0,
        f'recall@{n}': float(np.mean(recall_vals)) if recall_vals else 0.0,
        f'ndcg@{n}': float(np.mean(ndcg_vals)) if ndcg_vals else 0.0,
        'num_users_evaluated': len(ndcg_vals)
    }

    return results

### Hyperparameters Sweep

In [ ]:
def hyperparameter_sweep(
    model_class,
    train_data: pd.DataFrame,
    val_data: pd.DataFrame,
    param_grid: dict,
    metric: str = 'rmse'
) -> Tuple[dict, float, dict]:
    print(f"\nHyperparameter Sweep for {model_class.__name__} | Metric: {metric.upper()}")
    
    param_names = list(param_grid.keys())
    param_values = list(param_grid.values())
    results = {}
    
    ranking_metrics = ['ndcg', 'precision', 'recall', 'f1']
    is_ranking = metric.lower() in ranking_metrics
    
    for param_combo in product(*param_values):
        params = dict(zip(param_names, param_combo))
        print(f"Testing {params}")
        
        model = model_class(**params)
        fit_signature = inspect.signature(model.fit)
        if 'val_data' in fit_signature.parameters:
            model.fit(train_data, val_data)
        else:
            model.fit(train_data)
        
        measure_set_size = val_data.shape[0]
        
        if metric == 'rmse':
            score_val = evaluate_rmse(model, val_data)
            score_train = evaluate_rmse(model, train_data[:measure_set_size])
            print(f"RMSE: val={score_val:.4f}, train={score_train:.4f}")
            results[str(params)] = {'params': params, metric: score_val}
        elif is_ranking:
            ranking_results = evaluate_ranking_metrics(model, val_data, train_data, n=10)
            p, r = ranking_results['precision@10'], ranking_results['recall@10']
            f1 = 2 * (p * r) / (p + r + 1e-10)
            ranking_results['f1@10'] = f1
            
            score_val = ranking_results.get(f'{metric}@10', f1 if metric == 'f1' else 0)
            results[str(params)] = {
                'params': params,
                'ndcg': ranking_results['ndcg@10'],
                'precision': p,
                'recall': r,
                'f1': f1,
                'num_users': ranking_results['num_users_evaluated']
            }
            print(f"Validation {metric.upper()}@10 = {score_val:.4f}")
        else:
            raise ValueError(f"Unknown metric: {metric}")
    
    if metric == 'rmse':
        best_key = min(results.keys(), key=lambda k: results[k]['rmse'])
        best_score = results[best_key]['rmse']
    else:
        best_key = max(results.keys(), key=lambda k: results[k][metric])
        best_score = results[best_key][metric]
    
    best_params = results[best_key]['params']
    print(f"\nBest params: {best_params} | Best {metric.upper()}: {best_score:.4f}\n")
    
    return best_params, best_score, results


def train_and_predict_best(
    model_class,
    train_data: pd.DataFrame,
    val_data: pd.DataFrame,
    test_data: pd.DataFrame,
    param_grid: dict,
    output_filepath: str,
    metric: str = 'rmse',
    model_name: Optional[str] = None
) -> Tuple[object, pd.DataFrame]:
    best_params, best_score, _ = hyperparameter_sweep(
        model_class, train_data, val_data, param_grid, metric
    )
    
    if model_name is None:
        model_name = model_class.__name__
    
    print(f"Training final model with best params: {best_params}")
    
    final_model = model_class(**best_params)
    fit_signature = inspect.signature(final_model.fit)
    if 'val_data' in fit_signature.parameters:
        final_model.fit(train_data, val_data)
    else:
        final_model.fit(train_data)
    
    predictions_df = save_predictions_to_csv(
        model=final_model,
        test_data=test_data,
        output_filepath=output_filepath,
        model_name=f"{model_name} {best_params}"
    )
    
    print(f"Done. Best {metric.upper()}: {best_score:.4f} | Saved to: {output_filepath}")
    return final_model, predictions_df


# Task 1) + Task 2) 
# Implementation of different recommendation models as well as a hybrid model combining those recommendation models
# Experiments for both rating prediction and ranking tasks, and conducting offline evaluation

### Content-based model

In [ ]:
class ContentBasedCF:
    
    def __init__(self,
                 tfidf_max_features: int = 20000,
                 svd_dim: int = None,
                 normalize_emb: bool = True,
                 text_source: str = "full_text",
                 profile_agg: str = "weighted_avg",
                 positive_threshold: float = 4.0):
        self.tfidf_max_features = tfidf_max_features
        self.svd_dim = svd_dim
        self.normalize_emb = normalize_emb
        self.text_source = text_source
        self.profile_agg = profile_agg
        self.positive_threshold = positive_threshold

        self.train_data = None
        self.items = None
        self.users = None

        self.global_mean = None
        self.item_means = None
        self.user_means = None

        self.movies_df = None
        self.item_to_idx = None
        self.idx_to_item = None
        self.item_embeddings = None

        self._user_profiles = {}

    def _prepare_movie_features(self, movies_df: pd.DataFrame) -> pd.DataFrame:
        df = movies_df.copy()
        df['title'] = df['title'].fillna('')
        df['genres'] = df['genres'].fillna('')
        df['description'] = df['description'].fillna('')
        df['titlegenres'] = (df['title'] + ' ' + df['genres']).str.lower()
        df['full_text'] = (df['titlegenres'] + ' ' + df['description']).str.lower()
        return df

    def _tokenizer(self, s: str) -> List[str]:
        return re.findall(r"\w+|\S", s)

    def _compute_item_embeddings(self, corpus: List[str]) -> np.ndarray:
        tfidf = TfidfVectorizer(
            tokenizer=self._tokenizer,
            lowercase=False,
            ngram_range=(1, 2),
            stop_words='english',
            max_features=self.tfidf_max_features
        )
        X = tfidf.fit_transform(corpus)
        if self.svd_dim is not None and 0 < self.svd_dim < X.shape[1]:
            svd = TruncatedSVD(n_components=self.svd_dim, random_state=42)
            X = svd.fit_transform(X)
        else:
            X = X.toarray()
        if self.normalize_emb:
            X = normalize(X)
        return X

    def fit(self, train_data: pd.DataFrame, movies_df: pd.DataFrame):
        self.train_data = train_data
        self.users = set(train_data['user_id'].unique())
        self.items = set(train_data['item_id'].unique())
        self.global_mean = float(train_data['rating'].mean())
        self.item_means = train_data.groupby('item_id')['rating'].mean().to_dict()
        self.user_means = train_data.groupby('user_id')['rating'].mean().to_dict()

        self.movies_df = self._prepare_movie_features(movies_df)

        if self.text_source not in {'description', 'titlegenres', 'full_text'}:
            raise ValueError(f"Unknown text_source={self.text_source}")
        text_col = self.text_source

        self.item_to_idx = {iid: idx for idx, iid in enumerate(self.movies_df['item_id'])}
        self.idx_to_item = {idx: iid for iid, idx in self.item_to_idx.items()}

        self.item_embeddings = self._compute_item_embeddings(self.movies_df[text_col].tolist())
        self._user_profiles.clear()
        return self

    def _build_user_profile(self, user_id: int):
        rows = self.train_data[self.train_data['user_id'] == user_id]
        if rows.empty:
            return None

        if self.profile_agg == 'avg_pos':
            rows = rows[rows['rating'] >= self.positive_threshold]
            if rows.empty:
                return None

        rows = rows[rows['item_id'].isin(self.item_to_idx.keys())]
        if rows.empty:
            return None

        idxs = [self.item_to_idx[i] for i in rows['item_id']]
        embs = self.item_embeddings[idxs]

        if self.profile_agg == 'weighted_avg':
            weights = rows['rating'].to_numpy(dtype=float)
            prof = np.average(embs, axis=0, weights=weights)
        else:
            prof = embs.mean(axis=0)

        prof = prof / (np.linalg.norm(prof) + 1e-9)
        return prof

    def _get_user_profile(self, user_id: int):
        if user_id in self._user_profiles:
            return self._user_profiles[user_id]
        prof = self._build_user_profile(user_id)
        if prof is not None:
            self._user_profiles[user_id] = prof
        return prof

    def predict_rating(self, user_id: int, item_id: int) -> float:
        if self.train_data is None or self.item_embeddings is None:
            return float(np.clip(self.global_mean if self.global_mean is not None else 3.0, 1.0, 5.0))

        user_known = user_id in self.user_means
        item_known = item_id in self.item_to_idx

        if not user_known and not item_known:
            return float(np.clip(self.global_mean, 1.0, 5.0))
        if not user_known and item_known:
            return float(np.clip(self.item_means.get(item_id, self.global_mean), 1.0, 5.0))
        if user_known and not item_known:
            return float(np.clip(self.user_means.get(user_id, self.global_mean), 1.0, 5.0))

        profile = self._get_user_profile(user_id)
        if profile is None:
            um = self.user_means.get(user_id, self.global_mean)
            im = self.item_means.get(item_id, np.nan)
            return float(np.clip(um if np.isnan(im) else (um + im) / 2, 1.0, 5.0))

        item_vec = self.item_embeddings[self.item_to_idx[item_id]]
        item_vec = item_vec / (np.linalg.norm(item_vec) + 1e-9)

        sim = float(np.dot(profile, item_vec))
        um = self.user_means.get(user_id, self.global_mean)
        pred = um + 2.0 * sim
        return float(np.clip(pred, 1.0, 5.0))

    def recommend_topk(self, target_user: int, n: int = 10) -> List[Tuple[int, float]]:
        if self.train_data is None or self.item_embeddings is None:
            return []
        if target_user not in self.users:
            avail = [(iid, self.item_means.get(iid, self.global_mean))
                     for iid in self.item_to_idx.keys()]
            return sorted(avail, key=lambda x: x[1], reverse=True)[:n]

        seen = set(self.train_data[self.train_data['user_id'] == target_user]['item_id'])
        candidates = [iid for iid in self.item_to_idx.keys() if iid not in seen]
        if not candidates:
            return []

        prof = self._get_user_profile(target_user)
        if prof is None:
            avail = [(iid, self.item_means.get(iid, self.global_mean)) for iid in candidates]
            return sorted(avail, key=lambda x: x[1], reverse=True)[:n]

        idxs = [self.item_to_idx[iid] for iid in candidates]
        cand = self.item_embeddings[idxs]
        cand = cand / (np.linalg.norm(cand, axis=1, keepdims=True) + 1e-9)
        sims = cand @ prof

        base = self.user_means.get(target_user, self.global_mean)
        scores = np.clip(base + 2.0 * sims, 1.0, 5.0)
        top = np.argsort(scores)[::-1][:n]
        return [(candidates[i], float(scores[i])) for i in top]


### Fit Content-based for Rating

In [ ]:
param_grid = {
    'tfidf_max_features': [5000, 7500],
    'svd_dim': [256, None],
    'normalize_emb': [True],
    'text_source': ['titlegenres'],
    'profile_agg': ['avg'],
    'positive_threshold': [2.75, 3]
}

class ContentBasedCFWithMovies(ContentBasedCF):
    def __init__(self, movies_df, **kwargs):
        super().__init__(**kwargs)
        self._movies_df_external = movies_df
    
    def fit(self, train_data: pd.DataFrame):
        return super().fit(train_data, self._movies_df_external)

content_model_best, content_predictions = train_and_predict_best(
    model_class=lambda **kw: ContentBasedCFWithMovies(movies_df=movies, **kw),
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid=param_grid,
    output_filepath='../outputs/preds/predictions_content_best.csv',
    metric='rmse'
)

### Fit Content-based for Ranking

In [ ]:
content_ndcg_model, content_ndcg_predictions = train_and_predict_best(
    model_class=lambda **kw: ContentBasedCFWithMovies(movies_df=movies, **kw),
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'tfidf_max_features': [7500, 10000],
        'svd_dim': [1024, None],
        'normalize_emb': [True],
        'text_source': ['titlegenres'],
        'profile_agg': ['avg'],
        'positive_threshold': [2]
    },
    output_filepath='../outputs/preds/predictions_content_ndcg.csv',
    metric='ndcg',
    model_name='Content-Based CF (NDCG-optimized)'
)

content_ndcg_ranking_metrics = evaluate_ranking_metrics(
    model=content_ndcg_model,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=2.0
)

print(f"\nRanking Performance (NDCG-optimized):")
print(f"  Precision@10: {content_ndcg_ranking_metrics['precision@10']:.4f}")
print(f"  Recall@10:    {content_ndcg_ranking_metrics['recall@10']:.4f}")
print(f"  NDCG@10:      {content_ndcg_ranking_metrics['ndcg@10']:.4f}")


### User-based (UserKNN)

In [ ]:
class UserBasedCF:
    def __init__(self, k: int = 50, min_common: int = 2, use_pearson: bool = True):
        self.k = k
        self.min_common = min_common
        self.use_pearson = use_pearson
        self.train_data = None
        self.users = None
        self.items = None
        self.global_mean = None
        self.user_means = None
        self.item_means = None
        self.user_to_index = {}
        self.item_to_index = {}
        self.index_to_user = {}
        self.index_to_item = {}
        self.user_item_matrix = None
        self.user_similarity_matrix = None
        self.user_neighbors = {}
        self.all_predictions = None

    def _build_matrix(self, df: pd.DataFrame) -> csr_matrix:
        users = df["user_id"].unique()
        items = df["item_id"].unique()
        self.users, self.items = users, items
        self.user_to_index = {u: i for i, u in enumerate(users)}
        self.item_to_index = {it: j for j, it in enumerate(items)}
        self.index_to_user = {i: u for u, i in self.user_to_index.items()}
        self.index_to_item = {j: it for it, j in self.item_to_index.items()}
        rows = df["user_id"].map(self.user_to_index).to_numpy()
        cols = df["item_id"].map(self.item_to_index).to_numpy()
        vals = df["rating"].astype(float).to_numpy()
        return csr_matrix((vals, (rows, cols)), shape=(len(users), len(items)))

    def compute_user_similarity_matrix(self, R: csr_matrix) -> np.ndarray:
        X = R.toarray().astype(float)
        if self.use_pearson:
            mask = (X != 0)
            sums = X.sum(axis=1)
            counts = mask.sum(axis=1)
            means = np.divide(sums, counts, out=np.zeros_like(sums), where=counts != 0)
            X = np.where(mask, X - means[:, np.newaxis], 0.0)
        sims = cosine_similarity(X)
        np.fill_diagonal(sims, 1.0)
        return sims

    def build_topk_neighbors(self, sim_matrix: np.ndarray, k: int) -> Dict[int, List[Tuple[int, float]]]:
        topk = {}
        n_users = sim_matrix.shape[0]
        for u in range(n_users):
            sims = sim_matrix[u]
            top_idx = np.argpartition(-sims, range(1, k + 1))[1:k + 1]
            sorted_idx = top_idx[np.argsort(-sims[top_idx])]
            topk[self.index_to_user[u]] = [(self.index_to_user[i], float(sims[i])) for i in sorted_idx if sims[i] > 0]
        return topk

    def fit(self, train_data: pd.DataFrame):
        self.train_data = train_data
        self.global_mean = float(train_data["rating"].mean())
        self.user_means = train_data.groupby("user_id")["rating"].mean().to_dict()
        self.item_means = train_data.groupby("item_id")["rating"].mean().to_dict()
        self.user_item_matrix = self._build_matrix(train_data)
        self.user_similarity_matrix = self.compute_user_similarity_matrix(self.user_item_matrix)
        self.user_neighbors = self.build_topk_neighbors(self.user_similarity_matrix, self.k)
        self._precompute_all_predictions()
        return self

    def _precompute_all_predictions(self):
        R = self.user_item_matrix.toarray().astype(float)
        M = (R != 0).astype(float)
        sims = self.user_similarity_matrix.copy()
        sums = R.sum(axis=1)
        counts = M.sum(axis=1)
        user_means = np.divide(sums, counts, out=np.zeros_like(sums), where=counts > 0)
        n_users, n_items = R.shape
        preds = np.zeros((n_users, n_items), dtype=float)

        for u in range(n_users):
            sim_row = sims[u].copy()
            sim_row[u] = 0.0
            pos_mask = (sim_row > 0)
            if not np.any(pos_mask):
                preds[u, :] = user_means[u]
                continue
            pos_idx = np.where(pos_mask)[0]
            k_eff = min(self.k, pos_idx.size)
            top_idx = pos_idx[np.argpartition(-sim_row[pos_idx], k_eff - 1)[:k_eff]]
            w = sim_row[top_idx][:, None]
            R_top = R[top_idx, :]
            M_top = M[top_idx, :]
            mu_top = user_means[top_idx][:, None]
            centered = (R_top - mu_top) * M_top
            num = (w * centered).sum(axis=0)
            den = (np.abs(w) * M_top).sum(axis=0)
            den = np.where(den > 0, den, 1e-8)
            preds[u, :] = user_means[u] + num / den

        self.all_predictions = np.clip(preds, 1.0, 5.0)

    def predict_rating(self, user_id: int, item_id: int) -> float:
        if (user_id not in self.user_to_index) or (item_id not in self.item_to_index):
            return float(self.global_mean)
        u = self.user_to_index[user_id]
        i = self.item_to_index[item_id]
        return float(self.all_predictions[u, i])

    def recommend_topk(self, target_user: int, n: int = 10) -> List[Tuple[int, float]]:
        if target_user not in self.user_to_index:
            return sorted(self.item_means.items(), key=lambda x: x[1], reverse=True)[:n]
        u_idx = self.user_to_index[target_user]
        preds = self.all_predictions[u_idx].copy()
        seen_items = self.train_data.loc[self.train_data["user_id"] == target_user, "item_id"]
        seen_indices = seen_items.map(self.item_to_index).dropna().astype(int).to_numpy()
        preds[seen_indices] = -np.inf
        n_items = min(n, len(preds))
        if n_items == 0:
            return []
        top_idx = np.argpartition(-preds, n_items - 1)[:n_items]
        sorted_idx = top_idx[np.argsort(-preds[top_idx])]
        return [(self.index_to_item[i], float(preds[i])) for i in sorted_idx]


def train_user_based_cf(train_data: pd.DataFrame, k: int = 50, min_common: int = 2) -> UserBasedCF:
    model = UserBasedCF(k=k, min_common=min_common)
    model.fit(train_data)
    return model


### Fit UserKNN for Rating

In [ ]:
user_knn_model, user_knn_predictions = train_and_predict_best(
    model_class=UserBasedCF,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={'k': [200], 'min_common': [10]},
    output_filepath='../outputs/preds/predictions_userknn_best.csv',
    metric='rmse'
)

### Fit UserKNN for Ranking

In [ ]:
user_knn_ndcg_model, user_knn_ndcg_predictions = train_and_predict_best(
    model_class=UserBasedCF,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'k': [150, 250],
        'min_common': [2, 4]
    },
    output_filepath='../outputs/preds/predictions_userknn_ndcg.csv',
    metric='ndcg',
    model_name='User-Based CF (NDCG-optimized)'
)

user_knn_ndcg_ranking_metrics = evaluate_ranking_metrics(
    model=user_knn_ndcg_model,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=2.0
)

print(f"\nRanking Performance (NDCG-optimized):")
print(f"  Precision@10: {user_knn_ndcg_ranking_metrics['precision@10']:.4f}")
print(f"  Recall@10:    {user_knn_ndcg_ranking_metrics['recall@10']:.4f}")
print(f"  NDCG@10:      {user_knn_ndcg_ranking_metrics['ndcg@10']:.4f}")

### Item-based (ItemKNN)

In [ ]:
class ItemBasedCF:
    def __init__(self, k: int = 50, min_common: int = 2):
        self.k = k
        self.min_common = min_common
        self.train_data = None
        self.item_similarity_matrix = None
        self.users = None
        self.items = None
        self.user_means = None
        self.global_mean = None
        self.item_means = None

        self._user_ids = None
        self._item_ids = None
        self._user_index = None
        self._item_index = None
        self._R = None

    def _build_sparse_matrix(self, train_data: pd.DataFrame) -> csr_matrix:
        self._user_ids = train_data['user_id'].unique()
        self._item_ids = train_data['item_id'].unique()
        self._user_index = {u: i for i, u in enumerate(self._user_ids)}
        self._item_index = {it: j for j, it in enumerate(self._item_ids)}

        rows = train_data['user_id'].map(self._user_index).to_numpy()
        cols = train_data['item_id'].map(self._item_index).to_numpy()
        vals = train_data['rating'].astype(float).fillna(0.0).to_numpy()

        R = csr_matrix((vals, (rows, cols)), shape=(len(self._user_ids), len(self._item_ids)))
        return R

    def compute_item_similarity_matrix(self, train_data: pd.DataFrame) -> pd.DataFrame:
        self._R = self._build_sparse_matrix(train_data)
        sims = cosine_similarity(self._R.T, dense_output=False)
        item_similarity_matrix = pd.DataFrame(
            sims.toarray(), index=self._item_ids, columns=self._item_ids
        )
        np.fill_diagonal(item_similarity_matrix.values, 1.0)
        return item_similarity_matrix

    def get_k_item_neighbors(self, target_item: int, k: int = None) -> List[Tuple[int, float]]:
        if k is None:
            k = self.k
        if self.item_similarity_matrix is None or target_item not in self.item_similarity_matrix.index:
            return []
        
        sims = self.item_similarity_matrix.loc[target_item].drop(labels=target_item, errors='ignore')
        sims = sims[sims > 0]
        
        top_series = sims.sort_values(ascending=False).head(k)
        return list(top_series.items())

    def predict_rating(self, target_user: int, target_item: int) -> float:
        if self.item_similarity_matrix is None or self.train_data is None:
            return float(self.global_mean)

        if target_item not in self.item_similarity_matrix.index:
            return float(self.item_means.get(target_item, self.global_mean))

        user_ratings = self.train_data[self.train_data['user_id'] == target_user]
        if user_ratings.empty:
            return float(self.item_means.get(target_item, self.global_mean))

        user_ratings = user_ratings.set_index('item_id')['rating']

        sims = self.item_similarity_matrix.loc[target_item].drop(labels=target_item, errors='ignore')
        common_items = user_ratings.index.intersection(sims.index)
        
        if len(common_items) == 0:
            return float(self.item_means.get(target_item, self.global_mean))

        sims_common = sims.loc[common_items].dropna()
        
        if len(sims_common) < self.min_common:
            return float(self.item_means.get(target_item, self.global_mean))

        if sims_common.empty:
            return float(self.item_means.get(target_item, self.global_mean))

        top_sims = sims_common.sort_values(ascending=False).head(self.k)
        r_u = user_ratings.reindex(top_sims.index).astype(float)
        s = top_sims.reindex(r_u.index)
        
        mask = r_u.notna() & s.notna()
        if mask.sum() == 0:
            return float(self.item_means.get(target_item, self.global_mean))

        num = (s[mask] * r_u[mask]).sum()
        den = s[mask].abs().sum()

        if den == 0.0 or np.isnan(den):
            return float(self.item_means.get(target_item, self.global_mean))

        result = float(num / den)
        
        if np.isnan(result) or np.isinf(result):
            return float(self.item_means.get(target_item, self.global_mean))

        return float(np.clip(result, 1.0, 5.0))

    def recommend_topk(self, target_user: int, n: int = 10) -> List[Tuple[int, float]]:
        if self.train_data is None or self.item_similarity_matrix is None:
            return []
        
        if target_user not in self.train_data['user_id'].unique():
            popular_items = sorted(self.item_means.items(), key=lambda x: x[1], reverse=True)[:n]
            return popular_items 

        seen_items = set(self.train_data.loc[self.train_data['user_id'] == target_user, 'item_id'].unique())
        candidate_items = self.items - seen_items

        scores = []
        for item in candidate_items:
            pred = self.predict_rating(target_user, item)
            scores.append((item, float(pred)))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:n]

    def fit(self, train_data: pd.DataFrame):
        self.train_data = train_data
        self.users = set(train_data['user_id'].unique())
        self.items = set(train_data['item_id'].unique())
        self.global_mean = float(train_data['rating'].mean())
        self.user_means = train_data.groupby('user_id')['rating'].mean().to_dict()
        self.item_means = train_data.groupby('item_id')['rating'].mean().to_dict()

        self.item_similarity_matrix = self.compute_item_similarity_matrix(train_data)
        return self


### Fit ItemKNN for Rating

In [ ]:
item_knn_model, item_knn_predictions = train_and_predict_best(
    model_class=ItemBasedCF,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'k': [30, 50, 70],
        'min_common': [2, 5]
    },
    output_filepath='../outputs/preds/predictions_itemknn_best.csv',
    metric='rmse'
)

### Fir ItemKNN for Ranking

In [ ]:
print("\n" + "="*80)
print("ITEM-BASED CF: HYPERPARAMETER TUNING FOR NDCG@10")
print("="*80)

item_knn_ndcg_model, item_knn_ndcg_predictions = train_and_predict_best(
    model_class=ItemBasedCF,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'k': [60],
        'min_common': [2]
    },
    output_filepath='../outputs/preds/predictions_itemknn_ndcg.csv',
    metric='ndcg',
    model_name='Item-Based CF (NDCG-optimized)'
)


print("\n" + "="*80)
print("EVALUATING NDCG-OPTIMIZED ITEM-BASED MODEL")
print("="*80)

item_knn_ndcg_ranking_metrics = evaluate_ranking_metrics(
    model=item_knn_ndcg_model,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=2.0
)

print(f"\nRanking Performance (NDCG-optimized):")
print(f"  Precision@10: {item_knn_ndcg_ranking_metrics['precision@10']:.4f}")
print(f"  Recall@10:    {item_knn_ndcg_ranking_metrics['recall@10']:.4f}")
print(f"  NDCG@10:      {item_knn_ndcg_ranking_metrics['ndcg@10']:.4f}")

### Matrix Factorization

In [ ]:
class RatingDataset(Dataset):
    
    def __init__(self, user_ids, item_ids, ratings):
        self.users = torch.LongTensor(user_ids)
        self.items = torch.LongTensor(item_ids)
        self.ratings = torch.FloatTensor(ratings)
    
    def __len__(self):
        return len(self.ratings)
    
    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]


class MatrixFactorizationSGD:
    def __init__(self, n_factors, learning_rate, n_epochs, use_bias, eval_every, 
                 batch_size=1024, device='auto', l2_reg=0.01):
        self.n_factors = n_factors
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.use_bias = use_bias
        self.eval_every = eval_every
        self.batch_size = batch_size
        self.l2_reg = l2_reg
        
        if device == 'auto':
            self.device = torch.device('mps' if torch.backends.mps.is_available() 
                                      else 'cuda' if torch.cuda.is_available() 
                                      else 'cpu')
        else:
            self.device = torch.device(device)
        
        print(f"Device: {self.device}")
        
        self.P = None
        self.Q = None
        self.user_bias = None
        self.item_bias = None
        self.global_mean = None
        self.user_mapping = None
        self.item_mapping = None
        self.user_inv = None
        self.item_inv = None
        
        self.history = {
            'train_rmse': [],
            'val_rmse': [],
            'epochs': []
        }
        self.best_epoch = None
        self.best_val_rmse = float('inf')

    def _initialize_model(self, train_data: pd.DataFrame):
        self.user_mapping = {u: i for i, u in enumerate(train_data['user_id'].unique())}
        self.item_mapping = {i: j for j, i in enumerate(train_data['item_id'].unique())}
        self.user_inv = {i: u for u, i in self.user_mapping.items()}
        self.item_inv = {j: i for i, j in self.item_mapping.items()}

        n_users = len(self.user_mapping)
        n_items = len(self.item_mapping)

        self.P = nn.Embedding(n_users, self.n_factors).to(self.device)
        self.Q = nn.Embedding(n_items, self.n_factors).to(self.device)
        
        nn.init.normal_(self.P.weight, mean=0, std=0.01)
        nn.init.normal_(self.Q.weight, mean=0, std=0.01)

        if self.use_bias:
            self.user_bias = nn.Embedding(n_users, 1).to(self.device)
            self.item_bias = nn.Embedding(n_items, 1).to(self.device)
            nn.init.zeros_(self.user_bias.weight)
            nn.init.zeros_(self.item_bias.weight)
            self.global_mean = float(train_data['rating'].mean())

    def _prepare_training_data(self, train_data: pd.DataFrame):
        user_ids = [self.user_mapping[u] for u in train_data['user_id']]
        item_ids = [self.item_mapping[i] for i in train_data['item_id']]
        ratings = train_data['rating'].values
        
        return RatingDataset(user_ids, item_ids, ratings)

    def _train_one_epoch(self, dataloader, optimizer) -> float:
        self.P.train()
        self.Q.train()
        if self.use_bias:
            self.user_bias.train()
            self.item_bias.train()
        
        total_loss = 0.0
        n_batches = 0
        
        for users, items, ratings in dataloader:
            users = users.to(self.device)
            items = items.to(self.device)
            ratings = ratings.to(self.device)
            
            user_emb = self.P(users)
            item_emb = self.Q(items)
            
            preds = (user_emb * item_emb).sum(dim=1)
            
            if self.use_bias:
                preds = preds + self.global_mean
                preds = preds + self.user_bias(users).squeeze()
                preds = preds + self.item_bias(items).squeeze()
            
            mse_loss = nn.functional.mse_loss(preds, ratings)
            
            l2_loss = 0.0
            if self.l2_reg > 0:
                l2_loss = (
                    self.l2_reg * (user_emb.pow(2).sum() + item_emb.pow(2).sum())
                ) / users.size(0)
                
                if self.use_bias:
                    l2_loss += (
                        self.l2_reg * 0.1 * (
                            self.user_bias(users).pow(2).sum() + 
                            self.item_bias(items).pow(2).sum()
                        )
                    ) / users.size(0)
            
            loss = mse_loss + l2_loss
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += mse_loss.item()
            n_batches += 1
        
        return np.sqrt(total_loss / n_batches)

    def _should_evaluate(self, epoch: int) -> bool:
        return (epoch + 1) % self.eval_every == 0 or (epoch + 1) == self.n_epochs

    def _compute_rmse(self, data: pd.DataFrame) -> float:
        self.P.eval()
        self.Q.eval()
        if self.use_bias:
            self.user_bias.eval()
            self.item_bias.eval()
        
        preds = []
        trues = []
        
        with torch.no_grad():
            for _, row in data[['user_id', 'item_id', 'rating']].iterrows():
                pred = self.predict_rating(row['user_id'], row['item_id'])
                preds.append(float(pred))
                trues.append(float(row['rating']))
        
        return float(np.sqrt(mean_squared_error(trues, preds)))

    def _track_metrics(self, epoch: int, train_rmse: float, val_data: pd.DataFrame):
        self.history['train_rmse'].append(train_rmse)
        self.history['epochs'].append(epoch + 1)
        
        if val_data is not None:
            val_rmse = self._compute_rmse(val_data)
            self.history['val_rmse'].append(val_rmse)
            if val_rmse < self.best_val_rmse:
                self.best_val_rmse = val_rmse
                self.best_epoch = epoch + 1

    def fit(self, train_data: pd.DataFrame, val_data: pd.DataFrame):
        self.train_data = train_data
        self._initialize_model(train_data)
        dataset = self._prepare_training_data(train_data)
        dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        
        params = list(self.P.parameters()) + list(self.Q.parameters())
        if self.use_bias:
            params += list(self.user_bias.parameters()) + list(self.item_bias.parameters())
        
        optimizer = optim.Adam(params, lr=self.learning_rate)
        
        self.history = {'train_rmse': [], 'val_rmse': [], 'epochs': []}
        self.best_val_rmse = float('inf')
        self.best_epoch = None
        
        for epoch in tqdm(range(self.n_epochs), desc="Training MF"):
            train_rmse = self._train_one_epoch(dataloader, optimizer)
            
            if self._should_evaluate(epoch):
                self._track_metrics(epoch, train_rmse, val_data)
        
        if self.best_epoch is not None:
            print(f"Best val RMSE: {self.best_val_rmse:.4f} at epoch {self.best_epoch}")
        
        return self

    def predict_rating(self, user_id: int, item_id: int) -> float:
        if user_id not in self.user_mapping or item_id not in self.item_mapping:
            return self.global_mean if self.use_bias else 3.0
        
        self.P.eval()
        self.Q.eval()
        if self.use_bias:
            self.user_bias.eval()
            self.item_bias.eval()
        
        with torch.no_grad():
            u = torch.LongTensor([self.user_mapping[user_id]]).to(self.device)
            i = torch.LongTensor([self.item_mapping[item_id]]).to(self.device)
            
            user_emb = self.P(u)
            item_emb = self.Q(i)
            
            pred = torch.dot(user_emb[0], item_emb[0]).item()
            
            if self.use_bias:
                pred += self.global_mean
                pred += self.user_bias(u).item()
                pred += self.item_bias(i).item()
        
        return float(np.clip(pred, 1.0, 5.0))

    def recommend_topk(self, user_id: int, n: int = 10) -> List[Tuple[int, float]]:
        if user_id not in self.user_mapping:
            return []
        
        self.P.eval()
        self.Q.eval()
        if self.use_bias:
            self.user_bias.eval()
            self.item_bias.eval()
        
        with torch.no_grad():
            u = torch.LongTensor([self.user_mapping[user_id]]).to(self.device)
            user_emb = self.P(u)
            
            scores = torch.matmul(user_emb, self.Q.weight.T)[0]
            
            if self.use_bias:
                scores = scores + self.global_mean
                scores = scores + self.user_bias(u).item()
                scores = scores + self.item_bias.weight.squeeze()
            
            scores = scores.cpu().numpy()
        
        seen_items = self.train_data[self.train_data['user_id'] == user_id]['item_id'].values
        seen_idx = [self.item_mapping[i] for i in seen_items if i in self.item_mapping]
        scores[seen_idx] = -np.inf
        
        top_idx = np.argsort(scores)[::-1][:n]
        top_items = [self.item_inv[i] for i in top_idx]
        top_scores = scores[top_idx]
        
        return list(zip(top_items, top_scores))
    
    def get_training_history(self) -> Dict:
        return self.history
    
    def get_best_epoch(self) -> int:
        return self.best_epoch
    
    def plot_training_history(self, figsize=(12, 6), save_path=None):
        if not self.history['epochs']:
            print("No training history available.")
            return
        
        plt.figure(figsize=figsize)
        
        plt.plot(self.history['epochs'], self.history['train_rmse'], 
                label='Train RMSE', marker='o', linewidth=2, markersize=6)
        
        if self.history['val_rmse']:
            plt.plot(self.history['epochs'], self.history['val_rmse'], 
                    label='Validation RMSE', marker='s', linewidth=2, markersize=6)
            
            if self.best_epoch is not None:
                best_idx = self.history['epochs'].index(self.best_epoch)
                best_val_rmse = self.history['val_rmse'][best_idx]
                
                plt.axvline(x=self.best_epoch, color='red', linestyle='--', 
                           label=f'Best Epoch ({self.best_epoch})', alpha=0.7)
                plt.scatter([self.best_epoch], [best_val_rmse], 
                           color='red', s=150, zorder=5, marker='*')
                
                plt.annotate(f'Best: {best_val_rmse:.4f}',
                           xy=(self.best_epoch, best_val_rmse),
                           xytext=(10, 10), textcoords='offset points',
                           bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.7),
                           arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
        
        plt.xlabel('Epoch', fontsize=12)
        plt.ylabel('RMSE', fontsize=12)
        plt.title('Matrix Factorization Training History', fontsize=14, fontweight='bold')
        plt.legend(fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        
        plt.show()
        
        if self.history['val_rmse']:
            final_val_rmse = self.history['val_rmse'][-1]
            print(f"\nFinal Train: {self.history['train_rmse'][-1]:.4f}, Val: {final_val_rmse:.4f}")
            print(f"Best Val: {self.best_val_rmse:.4f} (Epoch {self.best_epoch})")
            
            if final_val_rmse > self.best_val_rmse:
                diff = final_val_rmse - self.best_val_rmse
                pct = (diff / self.best_val_rmse) * 100
                print(f"Overfitting: +{diff:.4f} ({pct:.2f}%)")


### Fit Matrix Factorization for Rating

In [ ]:
mf_model, mf_predictions = train_and_predict_best(
    model_class=MatrixFactorizationSGD,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'n_factors': [60, 80],
        'learning_rate': [0.002],
        'n_epochs': [40],
        'use_bias': [True],
        'eval_every': [5],
        'batch_size': [1024],
        'device': ['auto'],
        'l2_reg': [0.2]
    },
    output_filepath='../outputs/preds/predictions_mf_best.csv',
    metric='rmse'
)

mf_model.plot_training_history()
optimal_epochs = mf_model.get_best_epoch()

should_retrain = optimal_epochs is not None and optimal_epochs < mf_model.n_epochs

if should_retrain:
    print(f"\nRetraining with {optimal_epochs} epochs (best val: {mf_model.best_val_rmse:.4f})")

    mf_model = MatrixFactorizationSGD(
        n_factors=mf_model.n_factors,
        learning_rate=mf_model.learning_rate,
        n_epochs=optimal_epochs,
        use_bias=mf_model.use_bias,
        eval_every=mf_model.eval_every,
        batch_size=mf_model.batch_size,
        l2_reg=mf_model.l2_reg,
        device='auto'
    ).fit(train_data, val_data)

    print(f"Training complete: {optimal_epochs} epochs, Val RMSE: {mf_model.best_val_rmse:.4f}")

### Fit Matrix Factorization for Ranking

In [ ]:
mf_ndcg_model, mf_ndcg_predictions = train_and_predict_best(
    model_class=MatrixFactorizationSGD,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'n_factors': [75],
        'learning_rate': [0.01],
        'n_epochs': [20],
        'use_bias': [True],
        'eval_every': [30],
        'batch_size': [1024],
        'device': ['auto'],
        'l2_reg': [0.1]
    },
    output_filepath='../outputs/preds/predictions_mf_ndcg.csv',
    metric='ndcg',
    model_name='Matrix Factorization (NDCG-optimized)'
)

print("="*80)

mf_ndcg_ranking_metrics = evaluate_ranking_metrics(
    model=mf_ndcg_model,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=3.0
)

print(f"\nRanking Performance (NDCG-optimized):")
print(f"  Precision@10: {mf_ndcg_ranking_metrics['precision@10']:.4f}")
print(f"  Recall@10:    {mf_ndcg_ranking_metrics['recall@10']:.4f}")
print(f"  NDCG@10:      {mf_ndcg_ranking_metrics['ndcg@10']:.4f}")

### Bayesian Probabilistic Ranking

In [ ]:
class BPRDataset(Dataset):
    
    def __init__(self, triples):
        self.triples = torch.LongTensor(triples)
    
    def __len__(self):
        return len(self.triples)
    
    def __getitem__(self, idx):
        return self.triples[idx]


class BayesianProbabilisticRanking:
    """Bayesian Personalized Ranking with L2 regularization."""

    def __init__(self, n_factors, learning_rate, n_epochs, n_samples, eval_every, 
                 batch_size, device, l2_reg=0.01):
        self.n_factors = n_factors
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.n_samples = n_samples
        self.eval_every = eval_every
        self.batch_size = batch_size
        self.l2_reg = l2_reg
        
        if device == 'auto':
            self.device = torch.device('mps' if torch.backends.mps.is_available() 
                                      else 'cuda' if torch.cuda.is_available() 
                                      else 'cpu')
        else:
            self.device = torch.device(device)
        
        print(f"Using device: {self.device}")
        print(f"L2 Regularization: {self.l2_reg}")
        
        self.P = None
        self.Q = None
        self.user_bias = None
        self.item_bias = None
        self.user_mapping = None
        self.item_mapping = None
        self.user_inv = None
        self.item_inv = None
        self.global_mean = None
        self.user_means = None
        
        self.history = {
            'train_loss': [],
            'val_rmse': [],
            'epochs': []
        }
        self.best_epoch = None
        self.best_val_rmse = float('inf')

    def _generate_training_triples(self, ratings):
        triples = []
        user_items = ratings.groupby('user_id')['item_id'].apply(set).to_dict()
        all_items = set(ratings['item_id'].unique())

        for user_id, rated_items in user_items.items():
            user_idx = self.user_mapping[user_id]
            unrated_items = list(all_items - rated_items)
            if len(unrated_items) == 0:
                continue
            
            rated_items_list = list(rated_items)
            for pos_item_id in rated_items_list:
                pos_item_idx = self.item_mapping[pos_item_id]
                n_neg = min(self.n_samples, len(unrated_items))
                neg_item_ids = np.random.choice(unrated_items, size=n_neg, replace=False)
                for neg_item_id in neg_item_ids:
                    neg_item_idx = self.item_mapping[neg_item_id]
                    triples.append([user_idx, pos_item_idx, neg_item_idx])
        
        return np.array(triples)

    def _should_evaluate(self, epoch):
        return (epoch + 1) % self.eval_every == 0 or (epoch + 1) == self.n_epochs or epoch == 0

    def _compute_rmse_gpu(self, data):
        self.P.eval()
        self.Q.eval()
        if self.user_bias is not None:
            self.user_bias.eval()
            self.item_bias.eval()
        
        user_ids = []
        item_ids = []
        true_ratings = []
        
        for _, row in data[['user_id', 'item_id', 'rating']].iterrows():
            user_ids.append(row['user_id'])
            item_ids.append(row['item_id'])
            true_ratings.append(row['rating'])
        
        preds = []
        batch_size = 1000
        for i in range(0, len(user_ids), batch_size):
            batch_users = user_ids[i:i+batch_size]
            batch_items = item_ids[i:i+batch_size]
            
            batch_preds = []
            for u, it in zip(batch_users, batch_items):
                pred = self.predict_rating(u, it)
                batch_preds.append(pred)
            
            preds.extend(batch_preds)
        
        preds = np.array(preds)
        true_ratings = np.array(true_ratings)
        
        return float(np.sqrt(mean_squared_error(true_ratings, preds)))

    def _track_metrics(self, epoch, avg_loss, val_data):
        self.history['train_loss'].append(avg_loss)
        self.history['epochs'].append(epoch + 1)
        
        if val_data is not None:
            val_rmse = self._compute_rmse_gpu(val_data)
            self.history['val_rmse'].append(val_rmse)
            
            if val_rmse < self.best_val_rmse:
                self.best_val_rmse = val_rmse
                self.best_epoch = epoch + 1

    def fit(self, train_data, val_data):
        self.train_data = train_data
        self.user_mapping = {u: i for i, u in enumerate(train_data['user_id'].unique())}
        self.item_mapping = {i: j for j, i in enumerate(train_data['item_id'].unique())}
        self.user_inv = {i: u for u, i in self.user_mapping.items()}
        self.item_inv = {j: i for i, j in self.item_mapping.items()}

        self.global_mean = float(train_data['rating'].mean())
        self.user_means = train_data.groupby('user_id')['rating'].mean().to_dict()
        print(f"Global mean: {self.global_mean:.4f}")
        
        n_users = len(self.user_mapping)
        n_items = len(self.item_mapping)

        self.P = torch.nn.Embedding(n_users, self.n_factors).to(self.device)
        self.Q = torch.nn.Embedding(n_items, self.n_factors).to(self.device)
        
        torch.nn.init.normal_(self.P.weight, mean=0, std=0.01)
        torch.nn.init.normal_(self.Q.weight, mean=0, std=0.01)

        self.user_bias = torch.nn.Embedding(n_users, 1).to(self.device)
        self.item_bias = torch.nn.Embedding(n_items, 1).to(self.device)
        torch.nn.init.zeros_(self.user_bias.weight)
        torch.nn.init.zeros_(self.item_bias.weight)

        optimizer = optim.Adam(
            list(self.P.parameters()) + 
            list(self.Q.parameters()) +
            list(self.user_bias.parameters()) +
            list(self.item_bias.parameters()),
            lr=self.learning_rate
        )

        self.history = {'train_loss': [], 'val_rmse': [], 'epochs': []}
        self.best_val_rmse = float('inf')
        self.best_epoch = None

        for epoch in tqdm(range(self.n_epochs), desc="Training BPR"):
            self.P.train()
            self.Q.train()
            self.user_bias.train()
            self.item_bias.train()
            
            triples = self._generate_training_triples(train_data)
            np.random.shuffle(triples)
            
            dataset = BPRDataset(triples)
            dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
            
            epoch_loss = 0.0
            n_batches = 0
            
            for batch in dataloader:
                batch = batch.to(self.device)
                users = batch[:, 0]
                pos_items = batch[:, 1]
                neg_items = batch[:, 2]
                
                user_emb = self.P(users)
                pos_item_emb = self.Q(pos_items)
                neg_item_emb = self.Q(neg_items)
                
                pos_scores = (user_emb * pos_item_emb).sum(dim=1)
                neg_scores = (user_emb * neg_item_emb).sum(dim=1)
                
                bpr_loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-10).mean()
                
                l2_loss = 0.0
                if self.l2_reg > 0:
                    l2_loss = (
                        self.l2_reg * (
                            user_emb.pow(2).sum() + 
                            pos_item_emb.pow(2).sum() + 
                            neg_item_emb.pow(2).sum()
                        )
                    ) / users.size(0)
                    
                    l2_loss += (
                        self.l2_reg * 0.1 * (
                            self.user_bias(users).pow(2).sum() +
                            self.item_bias(pos_items).pow(2).sum() +
                            self.item_bias(neg_items).pow(2).sum()
                        )
                    ) / users.size(0)
                
                loss = bpr_loss + l2_loss
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                epoch_loss += bpr_loss.item()
                n_batches += 1
            
            avg_loss = epoch_loss / n_batches if n_batches > 0 else 0.0
            
            if self._should_evaluate(epoch):
                self._track_metrics(epoch, avg_loss, val_data)
                if self.history['val_rmse']:
                    print(f"Epoch {epoch+1}/{self.n_epochs} - Loss: {avg_loss:.4f} - Val RMSE: {self.history['val_rmse'][-1]:.4f}")
        
        if self.best_epoch is not None:
            print(f"\nBest validation RMSE: {self.best_val_rmse:.4f} at epoch {self.best_epoch}")
        
        return self

    def predict_rating(self, user_id, item_id):
        """Predict rating using bias terms for better accuracy."""
        if user_id not in self.user_mapping or item_id not in self.item_mapping:
            if user_id in self.user_means:
                return float(self.user_means[user_id])
            return float(self.global_mean)
        
        self.P.eval()
        self.Q.eval()
        self.user_bias.eval()
        self.item_bias.eval()
        
        with torch.no_grad():
            u = torch.LongTensor([self.user_mapping[user_id]]).to(self.device)
            i = torch.LongTensor([self.item_mapping[item_id]]).to(self.device)
            
            user_emb = self.P(u)
            item_emb = self.Q(i)
            
            score = (
                self.global_mean +
                self.user_bias(u).item() +
                self.item_bias(i).item() +
                torch.dot(user_emb[0], item_emb[0]).item()
            )
            
            return float(np.clip(score, 1.0, 5.0))

    def recommend_topk(self, user_id, n=10):
        """Generate top-N recommendations for ranking tasks."""
        if user_id not in self.user_mapping:
            return []
        
        self.P.eval()
        self.Q.eval()
        self.user_bias.eval()
        self.item_bias.eval()
        
        with torch.no_grad():
            u = torch.LongTensor([self.user_mapping[user_id]]).to(self.device)
            user_emb = self.P(u)
            
            scores = torch.matmul(user_emb, self.Q.weight.T)[0]
            scores = scores + self.user_bias(u).item()
            scores = scores + self.item_bias.weight.squeeze()
            
            scores = scores.cpu().numpy()
        
        seen_items = self.train_data[self.train_data['user_id'] == user_id]['item_id'].values
        seen_idx = [self.item_mapping[i] for i in seen_items if i in self.item_mapping]
        scores[seen_idx] = -np.inf
        
        top_idx = np.argsort(scores)[::-1][:n]
        top_items = [self.item_inv[i] for i in top_idx]
        top_scores = scores[top_idx]
        
        return list(zip(top_items, top_scores))

    def get_training_history(self):
        return self.history
    
    def get_best_epoch(self):
        return self.best_epoch
    
    def plot_training_history(self, figsize=(12, 6), save_path=None):
        if not self.history['epochs']:
            print("No training history available. Train the model first.")
            return
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
        
        ax1.plot(self.history['epochs'], self.history['train_loss'], 
                label='Train Loss', marker='o', linewidth=2, markersize=6, color='blue')
        ax1.set_xlabel('Epoch', fontsize=12)
        ax1.set_ylabel('Loss', fontsize=12)
        ax1.set_title('BPR Training Loss', fontsize=14, fontweight='bold')
        ax1.legend(fontsize=10)
        ax1.grid(True, alpha=0.3)
        
        if self.history['val_rmse']:
            ax2.plot(self.history['epochs'], self.history['val_rmse'], 
                    label='Validation RMSE', marker='s', linewidth=2, markersize=6, color='green')
            
            if self.best_epoch is not None:
                best_idx = self.history['epochs'].index(self.best_epoch)
                best_val_rmse = self.history['val_rmse'][best_idx]
                
                ax2.axvline(x=self.best_epoch, color='red', linestyle='--', 
                           label=f'Best Epoch ({self.best_epoch})', alpha=0.7)
                ax2.scatter([self.best_epoch], [best_val_rmse], 
                           color='red', s=150, zorder=5, marker='*')
                
                ax2.annotate(f'Best: {best_val_rmse:.4f}',
                           xy=(self.best_epoch, best_val_rmse),
                           xytext=(10, 10), textcoords='offset points',
                           bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.7),
                           arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
            
            ax2.set_xlabel('Epoch', fontsize=12)
            ax2.set_ylabel('RMSE', fontsize=12)
            ax2.set_title('Validation RMSE', fontsize=14, fontweight='bold')
            ax2.legend(fontsize=10)
            ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Plot saved to: {save_path}")
        
        plt.show()
        
        if self.history['val_rmse']:
            final_val_rmse = self.history['val_rmse'][-1]
            print("\n" + "="*60)
            print("Training Summary:")
            print("="*60)
            print(f"Final Train Loss: {self.history['train_loss'][-1]:.4f}")
            print(f"Final Val RMSE: {final_val_rmse:.4f}")
            print(f"Best Val RMSE: {self.best_val_rmse:.4f} (Epoch {self.best_epoch})")
            
            if final_val_rmse > self.best_val_rmse:
                diff = final_val_rmse - self.best_val_rmse
                pct = (diff / self.best_val_rmse) * 100
                print(f"Overfitting detected: +{diff:.4f} ({pct:.2f}%)")
            else:
                print("No overfitting detected")
            print("="*60)


### Fit BPR for Rating

In [ ]:
bpr_model, bpr_predictions = train_and_predict_best(
    model_class=BayesianProbabilisticRanking,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'n_factors': [50, 70],
        'learning_rate': [0.001, 0.02],
        'n_epochs': [30],
        'n_samples': [5],
        'eval_every': [5],
        'batch_size': [512],
        'device': ['auto'],
        'l2_reg': [0.1]
    },
    output_filepath='../outputs/preds/predictions_bpr_best.csv',
    metric='rmse'
)

bpr_model.plot_training_history(figsize=(12, 6), save_path=None)
optimal_epochs_bpr = bpr_model.get_best_epoch()

should_retrain = (
    optimal_epochs_bpr is not None and 
    optimal_epochs_bpr < bpr_model.n_epochs
)

if should_retrain:
    print("\n" + "="*80)
    print("STEP 3: Retraining BPR with optimal number of epochs")
    print("="*80)
    print(f"Optimal epoch found: {optimal_epochs_bpr}")
    print(f"Best validation RMSE: {bpr_model.best_val_rmse:.4f}")
    print(f"\nRetraining model with {optimal_epochs_bpr} epochs...")

    bpr_final = BayesianProbabilisticRanking(
        n_factors=bpr_model.n_factors,
        learning_rate=bpr_model.learning_rate,
        n_epochs=optimal_epochs_bpr,
        n_samples=bpr_model.n_samples,
        eval_every=bpr_model.eval_every,
        batch_size=bpr_model.batch_size,
        l2_reg=bpr_model.l2_reg,
        device='auto'
    ).fit(train_data, val_data)

    print("\n" + "="*80)
    print("BPR Training completed!")
    print("="*80)
    print(f"Final model trained with {optimal_epochs_bpr} epochs")
    print(f"Validation RMSE: {bpr_final.best_val_rmse:.4f}")
    print("="*80)
else:
    print("\n" + "="*80)
    print("No retraining needed - using existing model")
    print("="*80)
    bpr_final = bpr_model

### Fit BPR for Ranking

In [ ]:
bpr_ndcg_model, bpr_ndcg_predictions = train_and_predict_best(
    model_class=BayesianProbabilisticRanking,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'n_factors': [32],
        'learning_rate': [0.01],
        'n_epochs': [30],
        'n_samples': [5],
        'eval_every': [40],
        'batch_size': [8192],
        'device': ['auto'],
        'l2_reg': [0.01]
    },
    output_filepath='../outputs/preds/predictions_bpr_ndcg.csv',
    metric='ndcg',
    model_name='BPR (NDCG-optimized)'
)


print("\n" + "="*80)
print("EVALUATING NDCG-OPTIMIZED BPR MODEL")
print("="*80)

bpr_ndcg_ranking_metrics = evaluate_ranking_metrics(
    model=bpr_ndcg_model,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=1.0
)

print(f"\nRanking Performance (NDCG-optimized):")
print(f"  Precision@10: {bpr_ndcg_ranking_metrics['precision@10']:.4f}")
print(f"  Recall@10:    {bpr_ndcg_ranking_metrics['recall@10']:.4f}")
print(f"  NDCG@10:      {bpr_ndcg_ranking_metrics['ndcg@10']:.4f}")



### Hybrid-model

In [ ]:
class HybridRecommender:
    """Hybrid recommender with learnable weights over pre-trained component models using Ridge Regression (L2)."""

    def __init__(self, models: Dict[str, object], weights: Optional[Dict[str, float]] = None,
                 prediction_files: Optional[Dict[str, str]] = None, alpha: float = 1.0):
        self.models = models
        self.prediction_files = prediction_files
        self.regression_model = None
        self.scaler = StandardScaler()
        self.alpha = alpha
        
        if weights is None:
            n_models = len(models)
            self.weights = {name: 1.0 / n_models for name in models.keys()}
        else:
            total = sum(weights.values())
            self.weights = {name: w / total for name, w in weights.items()}
        
        self.train_data = None
        self.global_mean = None
        self.user_means = None
        self.model_names = list(models.keys())
        self.cached_predictions = {}
        
        self._validate_models()

    def _validate_models(self):
        for name, model in self.models.items():
            if not hasattr(model, 'predict_rating'):
                raise ValueError(f"Model '{name}' must have 'predict_rating' method")
            
            if hasattr(model, 'user_mapping') and model.user_mapping is None:
                raise ValueError(f"Model '{name}' appears to be unfitted.")

    def _load_predictions_from_files(self, validation_data: pd.DataFrame):
        """Load predictions from CSV files."""        
        for model_name in self.model_names:
            if model_name not in self.prediction_files:
                raise ValueError(f"No prediction file specified for model '{model_name}'")
            
            filepath = self.prediction_files[model_name]
            if not os.path.exists(filepath):
                raise FileNotFoundError(f"Prediction file not found: {filepath}")
            
            pred_df = pd.read_csv(filepath)
            pred_dict = {}
            
            for _, row in pred_df.iterrows():
                key = (int(row['user_id']), int(row['item_id']))
                pred_dict[key] = float(row['predicted_rating'])
            
            self.cached_predictions[model_name] = pred_dict
            print(f"  ✓ Loaded {len(pred_dict)} predictions for {model_name}")

    def fit(self, train_data: pd.DataFrame, validation_data: Optional[pd.DataFrame] = None):
        """Learn optimal weights for pre-trained models."""
        self.train_data = train_data
        self.global_mean = train_data['rating'].mean()
        self.user_means = train_data.groupby('user_id')['rating'].mean().to_dict()
        
        if validation_data is not None:
            if self.prediction_files:
                self._load_predictions_from_files(validation_data)
            self._learn_weights(validation_data)
        
        return self

    def _learn_weights(self, validation_data: pd.DataFrame):
        """Learn weights via Ridge regression (L2 regularization) on validation predictions."""
        X, y = [], []
        valid_pairs, skipped_pairs = 0, 0

        for _, row in tqdm(validation_data.iterrows(), total=len(validation_data), desc="Collecting predictions"):
            user_id = int(row['user_id'])
            item_id = int(row['item_id'])
            true_rating = float(row['rating'])
            key = (user_id, item_id)
            
            predictions = []
            all_valid = True
            
            for model_name in self.model_names:
                if self.cached_predictions and model_name in self.cached_predictions:
                    pred = self.cached_predictions[model_name].get(key)
                    if pred is None:
                        all_valid = False
                        break
                    predictions.append(pred)
                else:
                    model = self.models[model_name]
                    try:
                        pred = model.predict_rating(user_id, item_id)
                        if pred is None or (isinstance(pred, float) and np.isnan(pred)):
                            pred = self.user_means.get(user_id, self.global_mean)
                        predictions.append(float(pred))
                    except Exception:
                        all_valid = False
                        break

            if all_valid and len(predictions) == len(self.models):
                X.append(predictions)
                y.append(true_rating)
                valid_pairs += 1
            else:
                skipped_pairs += 1

        print(f"\n  Valid pairs: {valid_pairs}")
        print(f"  Skipped pairs: {skipped_pairs}")

        if len(X) == 0:
            print("\n⚠ Warning: No valid predictions. Using equal weights.")
            return

        X = np.array(X)
        y = np.array(y)

        print(f"\nFeature matrix shape: {X.shape}")
        print(f"Target vector shape: {y.shape}")

        X_scaled = self.scaler.fit_transform(X)
        self.regression_model = Ridge(alpha=self.alpha, fit_intercept=True)
        self.regression_model.fit(X_scaled, y)
        
        learned_weights = self.regression_model.coef_
        intercept = self.regression_model.intercept_

        print(f"\n✓ Ridge Regression model trained (alpha={self.alpha})!")
        print(f"  Intercept (bias): {intercept:.4f}")
        print(f"  Raw coefficients: {learned_weights}")

        abs_weights = np.abs(learned_weights)
        if abs_weights.sum() > 0:
            normalized_weights = abs_weights / abs_weights.sum()
        else:
            normalized_weights = np.ones(len(learned_weights)) / len(learned_weights)

        self.weights = {
            name: float(weight)
            for name, weight in zip(self.model_names, normalized_weights)
        }

        print(f"\n✓ Learned weights (normalized):")
        for name, weight in self.weights.items():
            print(f"    {name:20s}: {weight:.4f} ({weight*100:.1f}%)")

        val_rmse = self._evaluate_on_data(X, y)
        print(f"\n  Validation RMSE: {val_rmse:.4f}")

    def _evaluate_on_data(self, X: np.ndarray, y: np.ndarray) -> float:
        """Evaluate RMSE on given data."""
        if self.regression_model is None:
            return float('inf')
        X_scaled = self.scaler.transform(X)
        predictions = self.regression_model.predict(X_scaled)
        predictions = np.clip(predictions, 1.0, 5.0)
        return float(np.sqrt(np.mean((predictions - y) ** 2)))

    def predict_rating(self, user_id: int, item_id: int) -> float:
        """Predict a rating using the hybrid model."""
        predictions = []
        
        for model_name, model in self.models.items():
            try:
                pred = model.predict_rating(user_id, item_id)
                
                if pred is None or (isinstance(pred, float) and np.isnan(pred)):
                    pred = self.user_means.get(user_id, self.global_mean)
                
                predictions.append(float(pred))
            except Exception:
                predictions.append(self.global_mean)

        if len(predictions) == 0:
            return self.global_mean

        if self.regression_model is not None:
            X = np.array([predictions])
            X_scaled = self.scaler.transform(X)
            pred = self.regression_model.predict(X_scaled)[0]
            return float(np.clip(pred, 1.0, 5.0))

        weights = [self.weights[name] for name in self.model_names]
        total_weight = sum(weights)
        if total_weight == 0:
            return float(np.mean(predictions))
        
        weighted_pred = sum(p * w for p, w in zip(predictions, weights)) / total_weight
        return float(np.clip(weighted_pred, 1.0, 5.0))

    def recommend_topk(self, user_id: int, n: int = 10) -> List[Tuple[int, float]]:
        """Generate top-N recommendations for a user."""
        if self.train_data is None:
            return []
        
        seen_items = set(self.train_data[self.train_data['user_id'] == user_id]['item_id'].unique())
        all_items = set(self.train_data['item_id'].unique())
        candidate_items = all_items - seen_items
        
        if not candidate_items:
            return []
        
        scores = []
        for item in candidate_items:
            pred = self.predict_rating(user_id, item)
            scores.append((item, float(pred)))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:n]

    def get_weights(self) -> Dict[str, float]:
        """Return current model weights."""
        return self.weights.copy()

    def get_feature_importance(self) -> pd.DataFrame:
        """Return learned coefficients and normalized weights."""
        if self.regression_model is None:
            return pd.DataFrame({
                'model': self.model_names,
                'weight': [self.weights[name] for name in self.model_names],
            })
        
        return pd.DataFrame({
            'model': self.model_names,
            'weight': [self.weights[name] for name in self.model_names],
        }).sort_values('weight', ascending=False)

#### Run prediction for every possible user-item pair for each model and save to csv files. Train HybridRecommender.

In [ ]:
content_val_preds = save_predictions_to_csv(
    model=content_model_best,
    test_data=val_data,
    output_filepath='../outputs/preds/predictions_content_val.csv',
    model_name='Content'
)

user_knn_val_preds = save_predictions_to_csv(
    model=user_knn_model,
    test_data=val_data,
    output_filepath='../outputs/preds/predictions_userknn_val.csv',
    model_name='UserKNN'
)

item_knn_val_preds = save_predictions_to_csv(
    model=item_knn_model,
    test_data=val_data,
    output_filepath='../outputs/preds/predictions_itemknn_val.csv',
    model_name='ItemKNN'
)

mf_val_preds = save_predictions_to_csv(
    model=mf_model,
    test_data=val_data,
    output_filepath='../outputs/preds/predictions_mf_val.csv',
    model_name='MF'
)

hybrid_model = HybridRecommender(
    models={
        'Content': content_model_best,
        'UserKNN': user_knn_model,
        'ItemKNN': item_knn_model,
        'MF': mf_model},
    prediction_files={
        'Content': '../outputs/preds/predictions_content_val.csv',
        'UserKNN': '../outputs/preds/predictions_userknn_val.csv',
        'ItemKNN': '../outputs/preds/predictions_itemknn_val.csv',
        'MF': '../outputs/preds/predictions_mf_val.csv'},
    alpha=1.0
)

hybrid_model.fit(train_data, validation_data=val_data)


print("\n" + "="*80)
print("LEARNED WEIGHTS:")
print("="*80)
weights_df = hybrid_model.get_feature_importance()
display(weights_df)

print("\n" + "="*80)
print("GENERATING PREDICTIONS ON TEST SET")
print("="*80)

hybrid_predictions = save_predictions_to_csv(
    model=hybrid_model,
    test_data=test_data,
    output_filepath='../outputs/preds/predictions_hybrid.csv',
    model_name='Hybrid Recommender'
)

print("\n" + "="*80)
print("EVALUATION")
print("="*80)
hybrid_rmse = evaluate_rmse(hybrid_model, test_data)
print(f"Hybrid Model RMSE: {hybrid_rmse:.4f}")

### Fit Hybrid for Ranking

In [ ]:
class OptimizedHybridRanking:
    def __init__(
        self,
        prediction_files: Dict[str, str],
        train_data: pd.DataFrame,
        relevance_threshold: float = 3.0,
        normalize_per_user: bool = True,
        n_dirichlet: int = 300,
        coord_iters: int = 10,
        coord_grid: Tuple[float, ...] = (0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0),
        k_eval: int = 10,
        random_state: Optional[int] = 42,
    ):
        self.prediction_files = prediction_files
        self.train_data = train_data
        self.relevance_threshold = relevance_threshold
        self.normalize_per_user = normalize_per_user
        self.n_dirichlet = n_dirichlet
        self.coord_iters = coord_iters
        self.coord_grid = coord_grid
        self.k_eval = int(k_eval)
        self.rng = np.random.default_rng(random_state)
        self.model_names = list(prediction_files.keys())
        self.weights = {name: 1.0 / len(self.model_names) for name in self.model_names}
        self.predictions_df: Optional[pd.DataFrame] = None
        self.model_cols: List[str] = []
        self.user_topk: Dict[int, List[Tuple[int, float]]] = {}
        self._seen_items_by_user = self.train_data.groupby("user_id")["item_id"].apply(set).to_dict()

    @staticmethod
    def _dcg_at_k(rels: np.ndarray, k: int) -> float:
        rels = np.asarray(rels[:k], dtype=float)
        if rels.size == 0:
            return 0.0
        discounts = 1.0 / np.log2(np.arange(2, 2 + rels.size))
        return float(np.sum((2.0 ** rels - 1.0) * discounts))

    @staticmethod
    def _project_simplex(v: np.ndarray) -> np.ndarray:
        v = np.asarray(v, dtype=float)
        n = v.size
        u = np.sort(v)[::-1]
        cssv = np.cumsum(u)
        rho = np.where(u - (cssv - 1) / (np.arange(n) + 1) > 0)[0][-1]
        theta = (cssv[rho] - 1) / (rho + 1.0)
        return np.maximum(v - theta, 0.0)

    @staticmethod
    def _fill_missing_scores_per_user_mean(df: pd.DataFrame, score_cols: List[str]) -> pd.DataFrame:
        for c in score_cols:
            col_global = df[c].mean()
            df[c] = df.groupby('user_id')[c].transform(lambda s: s.fillna(s.mean())).fillna(col_global)
        return df

    @staticmethod
    def _minmax_per_user(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
        def _mm(g):
            g = g.copy()
            for c in cols:
                v = g[c].values
                vmin, vmax = np.nanmin(v), np.nanmax(v)
                g[c] = (v - vmin) / (vmax - vmin) if vmax > vmin else 0.5
            return g
        return df.groupby('user_id', group_keys=False).apply(_mm)

    def _load_and_merge_predictions(self, val_data: pd.DataFrame) -> None:
        pairs = None
        for path in self.prediction_files.values():
            part = pd.read_csv(path, usecols=['user_id', 'item_id']).drop_duplicates()
            pairs = part if pairs is None else pairs.merge(part, on=['user_id', 'item_id'], how='outer')
        df = pairs.copy()
        for model_name, filepath in self.prediction_files.items():
            pred_df = pd.read_csv(filepath, usecols=['user_id', 'item_id', 'predicted_rating'])
            df = df.merge(pred_df.rename(columns={'predicted_rating': f'score__{model_name}'}), on=['user_id', 'item_id'], how='left')
        self.model_cols = [c for c in df.columns if c.startswith('score__')]
        df = self._fill_missing_scores_per_user_mean(df, self.model_cols)
        val_r = val_data[['user_id', 'item_id', 'rating']].copy()
        df = df.merge(val_r, on=['user_id', 'item_id'], how='left')
        df['rel'] = 0.0
        mask = df['rating'].notna()
        df.loc[mask, 'rel'] = (df.loc[mask, 'rating'] >= self.relevance_threshold).astype(float)
        if self.normalize_per_user:
            df = self._minmax_per_user(df, self.model_cols)
        self.predictions_df = df

    def _eval_ndcg(self, w: np.ndarray) -> float:
        df = self.predictions_df.copy()
        df['hybrid'] = df[self.model_cols].values @ w
        ndcgs = []
        k = self.k_eval
        for uid, g in df.groupby('user_id'):
            seen = self._seen_items_by_user.get(uid, set())
            g = g[~g['item_id'].isin(seen)]
            if g.empty:
                continue
            g_sorted = g.sort_values('hybrid', ascending=False)
            top = g_sorted.head(k)
            g_with = g[g['rating'].notna()]
            if g_with.empty:
                continue
            rels = []
            rated_map = dict(zip(g_with['item_id'].values, g_with['rel'].values))
            for iid in top['item_id'].values:
                rels.append(rated_map.get(iid, 0.0))
            dcg = self._dcg_at_k(np.array(rels, dtype=float), k)
            ideal_rels = np.sort(g_with['rel'].values)[::-1][:k]
            idcg = self._dcg_at_k(ideal_rels, k)
            if idcg > 0:
                ndcgs.append(dcg / idcg)
        return 0.0 if not ndcgs else float(np.mean(ndcgs))

    def fit_weights(self, val_data: pd.DataFrame) -> float:
        self._load_and_merge_predictions(val_data)
        m = len(self.model_cols)
        if m == 0:
            raise ValueError("No model score columns found.")
        best_w = np.ones(m) / m
        best_ndcg = self._eval_ndcg(best_w)
        for _ in tqdm(range(self.n_dirichlet), desc="Dirichlet search", leave=False):
            w0 = self.rng.dirichlet(np.ones(m))
            nd = self._eval_ndcg(w0)
            if nd > best_ndcg:
                best_ndcg, best_w = nd, w0
        w = best_w.copy()
        for _ in range(self.coord_iters):
            improved = False
            for j in range(m):
                base = w.copy()
                best_local = best_ndcg
                best_vec = w
                for g in self.coord_grid:
                    w_try = base.copy()
                    w_try[j] = g
                    w_try = self._project_simplex(w_try)
                    nd = self._eval_ndcg(w_try)
                    if nd > best_local:
                        best_local, best_vec = nd, w_try
                if best_local > best_ndcg:
                    best_ndcg, w = best_local, best_vec
                    improved = True
            if not improved:
                break
        self.weights = {c.replace('score__', ''): float(wi) for c, wi in zip(self.model_cols, w)}
        return best_ndcg

    def precompute_topk(self) -> None:
        if self.predictions_df is None:
            raise ValueError("Call fit_weights first.")
        w = np.array([self.weights[c.replace('score__', '')] for c in self.model_cols])
        df = self.predictions_df.copy()
        df['hybrid_score'] = df[self.model_cols].values @ w
        self.user_topk = {}
        k = self.k_eval
        for uid, g in tqdm(df.groupby('user_id'), desc="Precomputing top-k", leave=False):
            seen = self._seen_items_by_user.get(uid, set())
            g = g[~g['item_id'].isin(seen)]
            if g.empty:
                self.user_topk[uid] = []
                continue
            top = g.sort_values('hybrid_score', ascending=False).head(k)
            self.user_topk[uid] = list(zip(top['item_id'].tolist(), top['hybrid_score'].astype(float).tolist()))

    def recommend_topk(self, user_id: int, n: int = 10) -> List[Tuple[int, float]]:
        if n != self.k_eval:
            return self.user_topk.get(user_id, [])[:n]
        return self.user_topk.get(user_id, [])

    def predict_rating(self, user_id: int, item_id: int) -> float:
        if self.predictions_df is None:
            return 0.0
        row = self.predictions_df[(self.predictions_df['user_id'] == user_id) & (self.predictions_df['item_id'] == item_id)]
        if row.empty:
            return 0.0
        w = np.array([self.weights[c.replace('score__', '')] for c in self.model_cols])
        return float(row[self.model_cols].values @ w)

    def get_weights(self) -> Dict[str, float]:
        return self.weights.copy()
    

In [ ]:
prediction_files = {
    'Content': '../outputs/preds/all_predictions_content.csv',
    'UserKNN': '../outputs/preds/all_predictions_userknn.csv',
    'ItemKNN': '../outputs/preds/all_predictions_itemknn.csv',
    'MF':      '../outputs/preds/all_predictions_mf.csv',
    'BPR':     '../outputs/preds/all_predictions_bpr.csv',
}

hybrid = OptimizedHybridRanking(
    prediction_files=prediction_files,
    train_data=train_data,
    relevance_threshold=3.0,
    normalize_per_user=True,
    n_dirichlet=300,
    coord_iters=10,
    k_eval=10,
)

best_val_ndcg = hybrid.fit_weights(val_data)

hybrid.precompute_topk()
metrics = evaluate_ranking_metrics(
    model=hybrid,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=3.0
)
print(metrics)
print(hybrid.get_weights())


# Task 3) Implement baselines for both rating prediction and ranking tasks, and perform experiments with those baselines

### Rating baseline models

In [ ]:
class ItemAverageBaseline:
    """Baseline: Predict rating as average rating of target item."""
    
    def __init__(self):
        self.item_means = None
        self.global_mean = None
        self.train_data = None
    
    def fit(self, train_data: pd.DataFrame):
        """Compute item average ratings."""
        self.train_data = train_data
        self.global_mean = float(train_data['rating'].mean())
        self.item_means = train_data.groupby('item_id')['rating'].mean().to_dict()
        return self
    
    def predict_rating(self, user_id: int, item_id: int) -> float:
        """Predict rating as item average."""
        rating = self.item_means.get(item_id, self.global_mean)
        return float(np.clip(rating, 1.0, 5.0))


class MeanHybridRating:
    """Baseline: Average predictions from all component models."""
    
    def __init__(self, models: Dict[str, object]):
        self.models = models
        self.model_names = list(models.keys())
        self.train_data = None
        self.global_mean = None
        self.user_means = None
    
    def fit(self, train_data: pd.DataFrame):
        """Store training data statistics."""
        self.train_data = train_data
        self.global_mean = float(train_data['rating'].mean())
        self.user_means = train_data.groupby('user_id')['rating'].mean().to_dict()
        return self
    
    def predict_rating(self, user_id: int, item_id: int) -> float:
        """Predict rating as average of all component predictions."""
        predictions = []
        
        for model_name, model in self.models.items():
            try:
                pred = model.predict_rating(user_id, item_id)
                if pred is not None and not np.isnan(pred):
                    predictions.append(float(pred))
            except Exception:
                pass
        
        if len(predictions) == 0:
            return self.user_means.get(user_id, self.global_mean)
        
        avg_pred = float(np.mean(predictions))
        return float(np.clip(avg_pred, 1.0, 5.0))


In [ ]:
print("\n" + "="*80)
print("TRAINING BASELINE MODELS")
print("="*80)
print("\n--- Rating Prediction Baselines ---")

print("\n1. Item Average Baseline")
item_avg_model = ItemAverageBaseline()
item_avg_model.fit(train_data)

item_avg_predictions = save_predictions_to_csv(
    model=item_avg_model,
    test_data=test_data,
    output_filepath='../outputs/preds/predictions_item_average.csv',
    model_name='Item Average Baseline'
)

print("\n2. Mean Hybrid Baseline (Rating)")
mean_hybrid_rating = MeanHybridRating(
    models={
        'UserKNN': user_knn_model,
        'ItemKNN': item_knn_model,
        'MatrixFactorization': mf_model}
)
mean_hybrid_rating.fit(train_data)

mean_hybrid_predictions = save_predictions_to_csv(
    model=mean_hybrid_rating,
    test_data=test_data,
    output_filepath='../outputs/preds/predictions_mean_hybrid.csv',
    model_name='Mean Hybrid Baseline')

### Ranking baseline models

In [ ]:
class RandomRecommender:
    """Baseline: Recommend random items."""
    
    def __init__(self, seed: int = 10):
        self.seed = seed
        self.train_data = None
        random.seed(seed)
    
    def fit(self, train_data: pd.DataFrame):
        self.train_data = train_data
        return self

    def recommend_topk(self, user_id: int, n: int = 10) -> List[Tuple[int, float]]:
        """Recommend random items."""
        seen_items = set(self.train_data[self.train_data['user_id'] == user_id]['item_id'].unique())
        all_items = set(self.train_data['item_id'].unique())
        candidate_items = list(all_items - seen_items)
        
        if not candidate_items: return []
        n_select = min(n, len(candidate_items))
        selected_items = random.sample(candidate_items, n_select)
        scores = [(item, random.random()) for item in selected_items]
        return scores

class PopularityRecommender:
    """Baseline: Recommend most popular items (by number of ratings)."""
    def __init__(self):
        self.item_popularity = None
        self.train_data = None
    
    def fit(self, train_data: pd.DataFrame):
        """Compute item popularity (rating count) and average ratings."""
        self.train_data = train_data
        self.item_popularity = train_data.groupby('item_id').size().to_dict()
        return self

    
    def recommend_topk(self, user_id: int, n: int = 10) -> List[Tuple[int, float]]:
        """Recommend most popular items."""
        seen_items = set(self.train_data[self.train_data['user_id'] == user_id]['item_id'].unique())
        all_items = set(self.train_data['item_id'].unique())
        candidate_items = all_items - seen_items
        
        scores = [(item, float(self.item_popularity.get(item, 0))) 
                  for item in candidate_items]
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:n]


In [ ]:
print("\n" + "-"*80)
print("1. RANDOM RECOMMENDER")
print("-"*80)
random_model = RandomRecommender(seed=1)
random_model.fit(train_data)
a = random_model.recommend_topk(user_id=0, n=10)
print(a)
random_ranking_metrics = evaluate_ranking_metrics(
    model=random_model,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=3.0
)

print(f"\nRandom Recommender - Ranking Performance:")
print(f"  Precision@10: {random_ranking_metrics['precision@10']:.4f}")
print(f"  Recall@10:    {random_ranking_metrics['recall@10']:.4f}")
print(f"  NDCG@10:      {random_ranking_metrics['ndcg@10']:.4f}")


print("\n" + "-"*80)
print("2. POPULARITY RECOMMENDER")
print("-"*80)
popularity_model = PopularityRecommender()
popularity_model.fit(train_data)
popularity_ranking_metrics = evaluate_ranking_metrics(
    model=popularity_model,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=3.0
)

print(f"\nPopularity Recommender - Ranking Performance:")
print(f"  Precision@10: {popularity_ranking_metrics['precision@10']:.4f}")
print(f"  Recall@10:    {popularity_ranking_metrics['recall@10']:.4f}")
print(f"  NDCG@10:      {popularity_ranking_metrics['ndcg@10']:.4f}")

### Comparison of Rating and Ranking models

| **Model**                               |  **RMSE**  |   **MAE**  |
| :-------------------------------------- | :--------: | :--------: |
| **Content-Based**                       |   1.1762   |   0.9111   |
| **Bayesian Personalized Ranking (BPR)** |   1.1554   |   0.9063   |
| **Item Average Baseline**               |   1.0331   |   0.8277   |
| **Item-Based CF**                       |   1.0064   |   0.7863   |
| **User-Based CF**                       |   0.9675   |   0.7564   |
| **Matrix Factorization (MF)**           |   0.9403   |   0.7427   |
| **Mean Hybrid Baseline**                |   0.9399   |   0.7393   |
| **Hybrid Model (Rating)**               | **0.9250** | **0.7285** |


| **Model**                               | **Precision@10** | **Recall@10** | **NDCG@10** |
| :-------------------------------------- | :--------------: | :-----------: | :---------: |
| **Random Recommender**                  |      0.0161      |     0.0067    |    0.0114   |
| **Content-Based**                       |      0.1046      |     0.0386    |    0.0955   |
| **User-Based CF**                       |      0.0107      |     0.0023    |    0.1041   |
| **Matrix Factorization (MF)**           |      0.0791      |     0.0225    |    0.1443   |
| **Item-Based CF**                       |      0.0266      |     0.0030    |    0.1627   |
| **Bayesian Personalized Ranking (BPR)** |      0.2137      |     0.0930    |    0.1778   |
| **Popularity Recommender**              |      0.2231      |     0.1048    |    0.1842   |
| **Hybrid Model (Ranking)**              |    **0.3194**    |   **0.1224**  |  **0.3100** |


# Task 4) Analysis of recommendation models. Analyzing the coefficients of hybrid model and the success of recommendation models for different users' groups. 

| Model   | Weight (Rating) | Weight (Ranking) |
| ------- | --------------: | ---------------: |
| Content |         -0.1498 |           0.0870 |
| UserKNN |          0.1938 |           0.0270 |
| ItemKNN |          0.2853 |           0.1122 |
| MF      |          0.3585 |           0.0887 |
| BPR     |               — |           0.6851 |


| Model   | Rating | Ranking |
| ------- | -----: | ------: |
| Content | 15.17% |   8.70% |
| UserKNN | 19.63% |   2.70% |
| ItemKNN | 28.90% |  11.22% |
| MF      | 36.30% |   8.87% |
| BPR     |      — |  68.51% |


# Task 5) Evaluation of beyond accuracy

In [ ]:
import pickle
import joblib
from pathlib import Path

# ==============================================================================
# SAVE/LOAD UTILITIES FOR ALL MODELS
# ==============================================================================

def save_content_based_model(model: ContentBasedCF, filepath: str):
    """Save ContentBasedCF model."""
    save_dict = {
        'params': {
            'tfidf_max_features': model.tfidf_max_features,
            'svd_dim': model.svd_dim,
            'normalize_emb': model.normalize_emb,
            'text_source': model.text_source,
            'profile_agg': model.profile_agg,
            'positive_threshold': model.positive_threshold
        },
        'global_mean': model.global_mean,
        'item_means': model.item_means,
        'user_means': model.user_means,
        'item_to_idx': model.item_to_idx,
        'idx_to_item': model.idx_to_item,
        'item_embeddings': model.item_embeddings,
        'movies_df': model.movies_df
    }
    joblib.dump(save_dict, filepath)
    print(f"✓ Saved ContentBasedCF to {filepath}")

def save_hybrid_ranking(model: OptimizedHybridRanking, filepath: str):
    """Save OptimizedHybridRanking model."""
    save_dict = {
        'prediction_files': model.prediction_files,
        'relevance_threshold': model.relevance_threshold,
        'normalize_per_user': model.normalize_per_user,
        'n_dirichlet': model.n_dirichlet,
        'coord_iters': model.coord_iters,
        'coord_grid': model.coord_grid,
        'k_eval': model.k_eval,
        'weights': model.weights,
        'model_names': model.model_names,
        'predictions_df': model.predictions_df,
        'model_cols': model.model_cols,
        'user_topk': model.user_topk,
        'train_data': model.train_data
    }
    joblib.dump(save_dict, filepath)
    print(f"✓ Saved OptimizedHybridRanking to {filepath}")


def load_hybrid_ranking(filepath: str) -> OptimizedHybridRanking:
    """Load OptimizedHybridRanking model."""
    save_dict = joblib.load(filepath)
    
    model = OptimizedHybridRanking(
        prediction_files=save_dict['prediction_files'],
        train_data=save_dict['train_data'],
        relevance_threshold=save_dict['relevance_threshold'],
        normalize_per_user=save_dict['normalize_per_user'],
        n_dirichlet=save_dict['n_dirichlet'],
        coord_iters=save_dict['coord_iters'],
        coord_grid=save_dict['coord_grid'],
        k_eval=save_dict['k_eval']
    )
    
    model.weights = save_dict['weights']
    model.model_names = save_dict['model_names']
    model.predictions_df = save_dict['predictions_df']
    model.model_cols = save_dict['model_cols']
    model.user_topk = save_dict['user_topk']
    
    print(f"✓ Loaded OptimizedHybridRanking from {filepath}")
    return model

def load_content_based_model(filepath: str, movies_df: pd.DataFrame) -> ContentBasedCF:
    """Load ContentBasedCF model."""
    save_dict = joblib.load(filepath)
    
    model = ContentBasedCF(**save_dict['params'])
    model.global_mean = save_dict['global_mean']
    model.item_means = save_dict['item_means']
    model.user_means = save_dict['user_means']
    model.item_to_idx = save_dict['item_to_idx']
    model.idx_to_item = save_dict['idx_to_item']
    model.item_embeddings = save_dict['item_embeddings']
    model.movies_df = save_dict['movies_df']
    model._user_profiles = {}
    
    print(f"✓ Loaded ContentBasedCF from {filepath}")
    return model


def save_user_based_cf(model: UserBasedCF, filepath: str):
    """Save UserBasedCF model."""
    save_dict = {
        'params': {
            'k': model.k,
            'min_common': model.min_common,
            'use_pearson': model.use_pearson
        },
        'global_mean': model.global_mean,
        'user_means': model.user_means,
        'item_means': model.item_means,
        'user_to_index': model.user_to_index,
        'item_to_index': model.item_to_index,
        'index_to_user': model.index_to_user,
        'index_to_item': model.index_to_item,
        'user_similarity_matrix': model.user_similarity_matrix,
        'all_predictions': model.all_predictions,
        'train_data': model.train_data
    }
    joblib.dump(save_dict, filepath)
    print(f"✓ Saved UserBasedCF to {filepath}")

def load_user_based_cf(filepath: str) -> UserBasedCF:
    """Load UserBasedCF model."""
    save_dict = joblib.load(filepath)
    
    model = UserBasedCF(**save_dict['params'])
    model.train_data = save_dict['train_data']
    model.global_mean = save_dict['global_mean']
    model.user_means = save_dict['user_means']
    model.item_means = save_dict['item_means']
    model.user_to_index = save_dict['user_to_index']
    model.item_to_index = save_dict['item_to_index']
    model.index_to_user = save_dict['index_to_user']
    model.index_to_item = save_dict['index_to_item']
    model.user_similarity_matrix = save_dict['user_similarity_matrix']
    model.all_predictions = save_dict['all_predictions']
    model.users = set(model.user_to_index.keys())
    model.items = set(model.item_to_index.keys())
    
    print(f"✓ Loaded UserBasedCF from {filepath}")
    return model


def save_item_based_cf(model: ItemBasedCF, filepath: str):
    """Save ItemBasedCF model."""
    save_dict = {
        'params': {
            'k': model.k,
            'min_common': model.min_common
        },
        'global_mean': model.global_mean,
        'user_means': model.user_means,
        'item_means': model.item_means,
        'item_similarity_matrix': model.item_similarity_matrix,
        'train_data': model.train_data
    }
    joblib.dump(save_dict, filepath)
    print(f"✓ Saved ItemBasedCF to {filepath}")

def load_item_based_cf(filepath: str) -> ItemBasedCF:
    """Load ItemBasedCF model."""
    save_dict = joblib.load(filepath)
    
    model = ItemBasedCF(**save_dict['params'])
    model.train_data = save_dict['train_data']
    model.global_mean = save_dict['global_mean']
    model.user_means = save_dict['user_means']
    model.item_means = save_dict['item_means']
    model.item_similarity_matrix = save_dict['item_similarity_matrix']
    model.users = set(model.train_data['user_id'].unique())
    model.items = set(model.train_data['item_id'].unique())
    
    print(f"✓ Loaded ItemBasedCF from {filepath}")
    return model


def save_matrix_factorization(model: MatrixFactorizationSGD, filepath: str):
    """Save MatrixFactorization model."""
    save_dict = {
        'params': {
            'n_factors': model.n_factors,
            'learning_rate': model.learning_rate,
            'n_epochs': model.n_epochs,
            'use_bias': model.use_bias,
            'eval_every': model.eval_every,
            'batch_size': model.batch_size,
            'l2_reg': model.l2_reg
        },
        'P_weight': model.P.weight.cpu().detach().numpy(),
        'Q_weight': model.Q.weight.cpu().detach().numpy(),
        'user_bias_weight': model.user_bias.weight.cpu().detach().numpy() if model.use_bias else None,
        'item_bias_weight': model.item_bias.weight.cpu().detach().numpy() if model.use_bias else None,
        'global_mean': model.global_mean,
        'user_mapping': model.user_mapping,
        'item_mapping': model.item_mapping,
        'user_inv': model.user_inv,
        'item_inv': model.item_inv,
        'train_data': model.train_data,
        'history': model.history,
        'best_epoch': model.best_epoch,
        'best_val_rmse': model.best_val_rmse
    }
    joblib.dump(save_dict, filepath)
    print(f"✓ Saved MatrixFactorization to {filepath}")

def load_matrix_factorization(filepath: str, device: str = 'auto') -> MatrixFactorizationSGD:
    """Load MatrixFactorization model."""
    save_dict = joblib.load(filepath)
    
    model = MatrixFactorizationSGD(**save_dict['params'], device=device)
    
    n_users = len(save_dict['user_mapping'])
    n_items = len(save_dict['item_mapping'])
    
    model.P = nn.Embedding(n_users, model.n_factors).to(model.device)
    model.Q = nn.Embedding(n_items, model.n_factors).to(model.device)
    
    model.P.weight.data = torch.from_numpy(save_dict['P_weight']).float().to(model.device)
    model.Q.weight.data = torch.from_numpy(save_dict['Q_weight']).float().to(model.device)
    
    if model.use_bias:
        model.user_bias = nn.Embedding(n_users, 1).to(model.device)
        model.item_bias = nn.Embedding(n_items, 1).to(model.device)
        model.user_bias.weight.data = torch.from_numpy(save_dict['user_bias_weight']).float().to(model.device)
        model.item_bias.weight.data = torch.from_numpy(save_dict['item_bias_weight']).float().to(model.device)
    
    model.global_mean = save_dict['global_mean']
    model.user_mapping = save_dict['user_mapping']
    model.item_mapping = save_dict['item_mapping']
    model.user_inv = save_dict['user_inv']
    model.item_inv = save_dict['item_inv']
    model.train_data = save_dict['train_data']
    model.history = save_dict['history']
    model.best_epoch = save_dict['best_epoch']
    model.best_val_rmse = save_dict['best_val_rmse']
    
    print(f"✓ Loaded MatrixFactorization from {filepath}")
    return model


def save_bpr(model: BayesianProbabilisticRanking, filepath: str):
    """Save BPR model."""
    save_dict = {
        'params': {
            'n_factors': model.n_factors,
            'learning_rate': model.learning_rate,
            'n_epochs': model.n_epochs,
            'n_samples': model.n_samples,
            'eval_every': model.eval_every,
            'batch_size': model.batch_size,
            'l2_reg': model.l2_reg
        },
        'P_weight': model.P.weight.cpu().detach().numpy(),
        'Q_weight': model.Q.weight.cpu().detach().numpy(),
        'user_bias_weight': model.user_bias.weight.cpu().detach().numpy(),
        'item_bias_weight': model.item_bias.weight.cpu().detach().numpy(),
        'global_mean': model.global_mean,
        'user_means': model.user_means,
        'user_mapping': model.user_mapping,
        'item_mapping': model.item_mapping,
        'user_inv': model.user_inv,
        'item_inv': model.item_inv,
        'train_data': model.train_data,
        'history': model.history,
        'best_epoch': model.best_epoch,
        'best_val_rmse': model.best_val_rmse
    }
    joblib.dump(save_dict, filepath)
    print(f"✓ Saved BPR to {filepath}")

def load_bpr(filepath: str, device: str = 'auto') -> BayesianProbabilisticRanking:
    """Load BPR model."""
    save_dict = joblib.load(filepath)
    
    model = BayesianProbabilisticRanking(**save_dict['params'], device=device)
    
    n_users = len(save_dict['user_mapping'])
    n_items = len(save_dict['item_mapping'])
    
    model.P = torch.nn.Embedding(n_users, model.n_factors).to(model.device)
    model.Q = torch.nn.Embedding(n_items, model.n_factors).to(model.device)
    model.user_bias = torch.nn.Embedding(n_users, 1).to(model.device)
    model.item_bias = torch.nn.Embedding(n_items, 1).to(model.device)
    
    model.P.weight.data = torch.from_numpy(save_dict['P_weight']).float().to(model.device)
    model.Q.weight.data = torch.from_numpy(save_dict['Q_weight']).float().to(model.device)
    model.user_bias.weight.data = torch.from_numpy(save_dict['user_bias_weight']).float().to(model.device)
    model.item_bias.weight.data = torch.from_numpy(save_dict['item_bias_weight']).float().to(model.device)
    
    model.global_mean = save_dict['global_mean']
    model.user_means = save_dict['user_means']
    model.user_mapping = save_dict['user_mapping']
    model.item_mapping = save_dict['item_mapping']
    model.user_inv = save_dict['user_inv']
    model.item_inv = save_dict['item_inv']
    model.train_data = save_dict['train_data']
    model.history = save_dict['history']
    model.best_epoch = save_dict['best_epoch']
    model.best_val_rmse = save_dict['best_val_rmse']
    
    print(f"✓ Loaded BPR from {filepath}")
    return model


def save_hybrid_model(model: HybridRecommender, filepath: str):
    """Save HybridRecommender model."""
    save_dict = {
        'weights': model.weights,
        'model_names': model.model_names,
        'global_mean': model.global_mean,
        'user_means': model.user_means,
        'train_data': model.train_data,
        'alpha': model.alpha,
        'regression_coef': model.regression_model.coef_ if model.regression_model else None,
        'regression_intercept': model.regression_model.intercept_ if model.regression_model else None,
        'scaler_mean': model.scaler.mean_ if hasattr(model.scaler, 'mean_') else None,
        'scaler_scale': model.scaler.scale_ if hasattr(model.scaler, 'scale_') else None
    }
    joblib.dump(save_dict, filepath)
    print(f"✓ Saved HybridRecommender to {filepath}")

def load_hybrid_model(filepath: str, models: Dict[str, object]) -> HybridRecommender:
    """Load HybridRecommender model."""
    save_dict = joblib.load(filepath)
    
    model = HybridRecommender(models=models, alpha=save_dict['alpha'])
    model.weights = save_dict['weights']
    model.model_names = save_dict['model_names']
    model.global_mean = save_dict['global_mean']
    model.user_means = save_dict['user_means']
    model.train_data = save_dict['train_data']
    
    if save_dict['regression_coef'] is not None:
        model.regression_model = Ridge(alpha=model.alpha)
        model.regression_model.coef_ = save_dict['regression_coef']
        model.regression_model.intercept_ = save_dict['regression_intercept']
        
        if save_dict['scaler_mean'] is not None:
            model.scaler.mean_ = save_dict['scaler_mean']
            model.scaler.scale_ = save_dict['scaler_scale']
    
    print(f"✓ Loaded HybridRecommender from {filepath}")
    return model


# ==============================================================================
# SAVE ALL TRAINED MODELS
# ==============================================================================

# Update save_all_models function
def save_all_models():
    """Save all trained models to disk."""
    print("\n" + "="*80)
    print("SAVING ALL MODELS")
    print("="*80)
    
    Path('models').mkdir(exist_ok=True)
    
    # Content-Based
    save_content_based_model(content_model_best, '../outputs/models/content_rmse.pkl')
    save_content_based_model(content_ndcg_model, '../outputs/models/content_ndcg.pkl')
    
    # UserKNN
    save_user_based_cf(user_knn_model, '../outputs/models/userknn_rmse.pkl')
    save_user_based_cf(user_knn_ndcg_model, '../outputs/models/userknn_ndcg.pkl')
    
    # ItemKNN
    save_item_based_cf(item_knn_model, '../outputs/models/itemknn_rmse.pkl')
    save_item_based_cf(item_knn_ndcg_model, '../outputs/models/itemknn_ndcg.pkl')
    
    # Matrix Factorization
    save_matrix_factorization(mf_model, '../outputs/models/mf_rmse.pkl')
    save_matrix_factorization(mf_ndcg_model, '../outputs/models/mf_ndcg.pkl')
    
    # BPR
    save_bpr(bpr_model, '../outputs/models/bpr_rmse.pkl')
    save_bpr(bpr_ndcg_model, '../outputs/models/bpr_ndcg.pkl')
    
    # Hybrid (Rating)
    save_hybrid_model(hybrid_model, '../outputs/models/hybrid_rmse.pkl')
    
    # Hybrid (Ranking) - NEW
    save_hybrid_ranking(hybrid, '../outputs/models/hybrid_ranking.pkl')
    
    print("\n✓ All models saved successfully!")


# Update load_all_models function
def load_all_models():
    """Load all trained models from disk."""
    print("\n" + "="*80)
    print("LOADING ALL MODELS")
    print("="*80)
    
    # Content-Based
    content_model_loaded = load_content_based_model('../outputs/models/content_rmse.pkl', movies)
    content_ndcg_loaded = load_content_based_model('../outputs/models/content_ndcg.pkl', movies)
    
    # UserKNN
    userknn_model_loaded = load_user_based_cf('../outputs/models/userknn_rmse.pkl')
    userknn_ndcg_loaded = load_user_based_cf('../outputs/models/userknn_ndcg.pkl')
    
    # ItemKNN
    itemknn_model_loaded = load_item_based_cf('../outputs/models/itemknn_rmse.pkl')
    itemknn_ndcg_loaded = load_item_based_cf('../outputs/models/itemknn_ndcg.pkl')
    
    # Matrix Factorization
    mf_model_loaded = load_matrix_factorization('../outputs/models/mf_rmse.pkl')
    mf_ndcg_loaded = load_matrix_factorization('../outputs/models/mf_ndcg.pkl')
    
    # BPR
    bpr_model_loaded = load_bpr('../outputs/models/bpr_rmse.pkl')
    bpr_ndcg_loaded = load_bpr('../outputs/models/bpr_ndcg.pkl')
    
    # Hybrid (Rating)
    component_models = {
        'Content': content_model_loaded,
        'UserKNN': userknn_model_loaded,
        'ItemKNN': itemknn_model_loaded,
        'MF': mf_model_loaded
    }
    hybrid_model_loaded = load_hybrid_model('../outputs/models/hybrid_rmse.pkl', component_models)
    
    # Hybrid (Ranking) - NEW
    hybrid_ranking_loaded = load_hybrid_ranking('../outputs/models/hybrid_ranking.pkl')
    
    print("\n✓ All models loaded successfully!")
    
    return {
        'content_rmse': content_model_loaded,
        'content_ndcg': content_ndcg_loaded,
        'userknn_rmse': userknn_model_loaded,
        'userknn_ndcg': userknn_ndcg_loaded,
        'itemknn_rmse': itemknn_model_loaded,
        'itemknn_ndcg': itemknn_ndcg_loaded,
        'mf_rmse': mf_model_loaded,
        'mf_ndcg': mf_ndcg_loaded,
        'bpr_rmse': bpr_model_loaded,
        'bpr_ndcg': bpr_ndcg_loaded,
        'hybrid_rating': hybrid_model_loaded,
        'hybrid_ranking': hybrid_ranking_loaded  # NEW
    }


# Test the ranking hybrid model
save_all_models()
loaded_models = load_all_models()
test_user = 1

# Get ranking recommendations
ranking_recs = loaded_models['hybrid_ranking'].recommend_topk(test_user, n=10)
print(f"\nTop-10 ranking recommendations for user {test_user}:")
for item_id, score in ranking_recs:
    print(f"  Item {item_id}: {score:.4f}")